In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from einops import rearrange
from typing import *
from abc import ABC, abstractmethod
import copy
import random
import numpy as np
from jaxtyping import Float, Bool, Int,Array
from torch import Tensor
from einops import rearrange, einsum
import einx
import math
from torch.func import vmap, jacrev
from tqdm import tqdm
import copy

# init

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)                       
    np.random.seed(seed)                   
    torch.manual_seed(seed)                

    if torch.backends.mps.is_available():
  
        print("Using MPS: Seed fixed for reproducibility")
    elif torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False      

In [4]:
set_seed(42)

Using MPS: Seed fixed for reproducibility


# Extractor base

In [5]:
class Extractor(nn.Module, ABC):
    def __init__(self, d_in: int, d_out: int, d_hidden: int, img_size: int):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.d_hidden = d_hidden
        self.img_size = img_size

    @abstractmethod
    def forward(self, x: Float[Array, "batch channel height width"]) -> Float[Array, "batch seq_len dim"]:
        pass

## cutomerized linear+conv

In [6]:

class Dense(nn.Module,ABC):
    def __init__(self):
        super().__init__()
        
    @staticmethod
    
    def trunc_normal_init(tensor: Float[Array, "..."], d_in: int, d_out: int):
        std=math.sqrt(2.0/ (d_in + d_out))
        nn.init.trunc_normal_(tensor, std=std, a=-3*std, b=3*std)
    
    @abstractmethod
    def forward(self,x:Float[Array, "... d_in"]) -> Float[Array, "... d_out"]:
        pass
    

class Linear(Dense):
    def __init__(self, d_in: int, d_out: int,bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty( d_out, d_in, requires_grad=True))
        self.trunc_normal_init(self.weight,d_out, d_in)
        if bias:
            self.bias = nn.Parameter(torch.zeros(d_out, requires_grad=True))
        else:
            self.bias = None
    
    def forward(self, x: Float[Array, "... d_in"]) -> Float[Array, "... d_out"]:
         return einsum(x,self.weight,"... d_in, d_out d_in-> ... d_out")+ (self.bias if self.bias is not None else 0)
     
class Conv2d(Dense):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, padding: int = 0, bias: bool = True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, kernel_size, kernel_size, requires_grad=True))
        self.trunc_normal_init(self.weight, in_channels * kernel_size * kernel_size, out_channels)
        
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels, requires_grad=True))
        else:
            self.bias = None
    
    def forward(self, x: Float[Array, "batch in_channels height width"]) -> Float[Array, "batch out_channels h w"]:
        return nn.functional.conv2d(x, self.weight, self.bias, stride=self.stride, padding=self.padding)

    
    



# minimal nn model

## minimal unet

In [15]:
    
class ResidualLayer(nn.Module):
    def __init__(self, channels_in:int):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.SiLU(),
            nn.BatchNorm2d(channels_in),
            nn.Conv2d(channels_in, channels_in, kernel_size=3, padding=1)
        )
        self.block2 = nn.Sequential(
            nn.SiLU(),
            nn.BatchNorm2d(channels_in),
            nn.Conv2d(channels_in, channels_in, kernel_size=3, padding=1)
        
        )
        
    def forward(self, x: Float[Array, "batch channels height width"]) -> Float[Array, "batch channels height width"]:
        res = x
        x= self.block1(x)
        x= self.block2(x)
        
        x+=res
        return x


class MiniUNet(Extractor):
    def __init__(self, d_in=1, d_out=8, d_hidden=4, kernel_num=4, img_size=32):
        super().__init__(d_in, d_out, d_hidden, img_size)
        self.init_conv = nn.Sequential(Conv2d(d_in, d_hidden, kernel_num, padding=1), 
                                       nn.BatchNorm2d(d_hidden), 
                                       nn.SiLU())
        
        self.res_blocks=ResidualLayer(d_hidden)
        self.conv2=Conv2d(d_hidden, d_out,kernel_size=3, stride=2, padding=1)
    def forward(self, x: Float[Array, "batch channel height width"]) -> Float[Array, "batch seq_len dim"]:
        x= self.init_conv(x)
        res=x
        x = self.res_blocks(x)
        x = self.conv2(x)
        x= rearrange(x, "b c h w -> b (h w) c")
        return x

## minimal vit

In [16]:
def scale_dot_product_attetnion(
    Query:Float[Array,"... queries d_k"],
    Key:Float[Array,"... keys d_k"],
    Value:Float[Array,"... values d_v"],
    mask:Bool[Array,"... queries keys"] | None = None,
)->Float[Array,"... queries d_v"]:
    d_k=Key.shape[-1]
    attention_score=einsum(Query,Key,"... queries d_k,... keys d_k-> ... queries keys")/math.sqrt(d_k)
    if mask is not None:
        attention_score=torch.where(mask, attention_score, torch.tensor(float('-inf')))
    attention_weights=torch.softmax(attention_score,dim=-1)
    
    return einsum(attention_weights,Value,"... query key, ... key d_v ->  ... query d_v")

class Embedding(nn.Module):
    def __init__(self,num_embeddings, embedding_dim, device=None, dtype=None):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.weight = nn.Parameter(nn.init.trunc_normal_(torch.empty(num_embeddings, embedding_dim), std=1.0,
                                 a=-3,b=3))
        
    def forward(self,token_ids:Int[Array,"..."])->Float[Array,"... d_model"]:
        return self.weight[token_ids,:]
class RMSNorm(nn.Module):
    def __init__(self,d_model:int,eps:float=1e-5,device=None, dtype=None):
        super().__init__()
        self.d_model = d_model
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model, device=device, dtype=dtype))
    
    def forward(self,x:Float[Array,"batch_size seq_len d_model"])->Float[Array,"batch_size ..."]:
        in_dtype=  x.dtype
        x=x.to(torch.float32)
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        
        return self.weight * (x *rms)
class RotaryEmbedding(nn.Module):
    def __init__(self, d_k: int, max_seq_len: int, theta: float = 10000.0):
        super().__init__()
        self.register_buffer(
            "_freq_cis_cache",
            RotaryEmbedding._init_cache(max_seq_len, d_k, theta), persistent=False
        )
    @staticmethod
    def _init_cache(max_seq_len:int, d_k:int, theta:float)-> Float[Array,"2 max_seq_len d_k/2"]:
        assert d_k % 2 == 0, "d_k must be even"
        d=torch.arange(0,d_k,2).float()
        t=torch.arange(0,max_seq_len).float()
        freqs= theta ** (-d / d_k)
        freqs= einsum(t,freqs,'t,f -> t f')
        cos,sin= torch.cos(freqs), torch.sin(freqs)
        return torch.stack((cos,sin))
    
    def forward(self,x:Float[Array,"... seq  d_k"],pos_ids: Int[Array, " ... seq"])->Float[Array,"...  seq d_k"]:
        x1, x2 = rearrange(x, "... (d r) -> ... d r", r=2).unbind(-1)
        cos, sin = einx.get_at('cos_sin [pos] half_dim, ... -> cos_sin ... half_dim', self._freq_cis_cache, pos_ids)
        x1_rot = cos * x1 - sin * x2
        x2_rot = sin * x1 + cos * x2
        result = einx.rearrange('... x_half, ... x_half -> ... (x_half (1 + 1))', x1_rot, x2_rot).contiguous()
        return result

class CustomizedAttention(nn.Module,ABC):
   def __init__(
       self,
       d_model:int,
       num_heads:int,
       positional_encoder:None,
   ):
       super().__init__()
       assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
       self.d_model = d_model
       self.num_heads = num_heads
       self.d_k = d_model // num_heads
       self.d_v=self.d_k
       
       self.q_proj=Linear(self.d_model,self.num_heads*self.d_k)
       self.k_proj=Linear(self.d_model,self.num_heads*self.d_k)
       self.v_proj=Linear(self.d_model,self.num_heads*self.d_v)
       
       self.out_proj=Linear(self.num_heads*self.d_v,self.d_model)
       
       self.positional_encoder = positional_encoder
       
   @abstractmethod
   def forward(
        self,
        x:Float[Array,"... sequence d_k"],
        token_positions:Int[Array,"... seq"] | None = None)->Float[Array,"... sequence d_v"]:
        pass 
    


class CausalMultiHeadAttention(CustomizedAttention):
    def __init__(self, d_model: int, num_heads: int, positional_encoder:  RotaryEmbedding):
        super().__init__(d_model, num_heads, positional_encoder)
    
    def forward(
        self,
        x: Float[Array, "... sequence d_k"],
        token_positions: Int[Array, "... seq"] | None = None
    ) -> Float[Array, "... sequence d_v"]:
        
        *b,seq_len,d_model=x.shape
        
        assert d_model ==self.d_model
        Q=self.q_proj(x)
        K=self.k_proj(x)
        V=self.v_proj(x)
        
        Q,K,V=(
            rearrange(X,"... sequence (heads d_k)->... heads sequence d_k" ,heads=self.num_heads)
            for X in (Q, K, V)
        )
       
        if token_positions is  None:
            token_positions = einx.rearrange("seq -> b... seq",torch.arange(seq_len, device=x.device),b=[1] * len(b))
        token_positions=rearrange(token_positions, "... seq -> ... 1 seq ")
        Q=self.positional_encoder(Q,token_positions)
        K=self.positional_encoder(K,token_positions)
        
        #seq=torch.arange(seq_len, device=x.device)
        #qi=einx.rearrange("query ->b... 1 query 1",seq,b=[1]*len(b))
        #ki=einx.rearrange("key -> b... 1  1 key ",seq,b=[1]*len(b))
        #casual_mask=qi >=ki
        
        attn_output= scale_dot_product_attetnion(
            Q, K, V, mask=None
        )
        
        attn_output=rearrange(attn_output,"... h seq d_v -> ... seq (h d_v)").contiguous()
        output=self.out_proj(attn_output)
        return output
    
    
class MiniVisionTransformer(Extractor):
    def __init__(self, d_in=1, d_out=8, d_hidden=16, patch_size=4,num_heads=2,img_size=32):
        super().__init__(d_in, d_out, d_hidden, img_size)
        self.initconv=Conv2d(d_in,d_hidden, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_hidden))
        self.positions= Embedding((img_size // patch_size) ** 2 + 1, d_hidden)
        self.position_encoder = RotaryEmbedding(
    d_k=d_hidden // num_heads,
    max_seq_len=(img_size // patch_size) ** 2 + 1
)
 
        self.attn= CausalMultiHeadAttention(
            d_model=d_hidden,
            num_heads=num_heads,
            positional_encoder=self.position_encoder,
        )
        self.norm1 = RMSNorm(d_hidden)
        self.norm2 = RMSNorm(d_hidden)
        self.ffn = nn.Sequential(
            Linear(d_hidden, d_hidden * 4),
            nn.GELU(),
            Linear(d_hidden * 4, d_hidden)
        )
        self.proj= Linear(d_hidden, d_out)
        
    def forward(self, x: Float[Array, "batch channel height width"]) -> Float[Array, "batch seq_len dim"]:
        x= self.initconv(x)
        b, c, h, w = x.shape
        x = rearrange(x, "b c h w -> b (h w) c")
        cls_token = self.cls_token.expand(b, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        pos_ids=torch.arange(x.size(1), device=x.device)
        pos_ids=rearrange(pos_ids,'seq -> 1 seq')
        x+=self.positions(pos_ids)
        x_attn=self.attn(self.norm1(x))
        x_sub=x+x_attn
        x_ffn=self.ffn(self.norm2(x_sub))
        ffn_sub=x_sub+x_ffn
        
        return  self.proj(ffn_sub)
        
    
    

## mlp

In [17]:
class MLPExtractor(Extractor):
    def __init__(self, d_in=32*32, d_out=8, d_hidden=64, img_size=32):
        super().__init__(d_in, d_out, d_hidden, img_size)
        self.layer = nn.Sequential(
            Linear(d_in, d_hidden),
            nn.ReLU(),
            Linear(d_hidden, d_out)
        )

    def forward(self, x: Float[Array, "batch channel height width"]) -> Float[Array, "batch seq_len dim"]:
        
        #x = x.view(x.size(0), -1)  # (b, d_in)
        #x = self.layer(x).unsqueeze(1)  # (b, 1, d_out)
        x=rearrange(x, 'b c h w -> b (c h w)')
        x= self.layer(x)
        x= rearrange(x, 'b d_out -> b 1 d_out ')  # Reshape to (b, 1, d_out)
      
        
        return x

In [18]:
class PCAMLP(Extractor):
    def __init__(self, d_in=32*32, d_out=8, d_hidden=32, img_size=32, n_components=64):
        super().__init__(d_in, d_out, d_hidden, img_size)
        self.n_components = n_components
        self.fitted = False
        self.register_buffer("pca_components", torch.empty(self.n_components, d_in))
        self.register_buffer("mean", torch.empty(d_in))  # [d_in]

        self.linear = nn.Sequential(
            Linear(n_components, d_hidden),
            nn.ReLU(),
            Linear(d_hidden, d_out)
        )

    def _fit_pca_from_batch(self, x: torch.Tensor):
       x_flat = x.view(x.size(0), -1)  # [batch, d_in]
       mean = x_flat.mean(dim=0)
       x_centered = x_flat - mean

 
       U, S, Vh = torch.linalg.svd(x_centered, full_matrices=False)
       available_components = Vh.size(0)  # ≤ batch_size

 
       used_components = min(self.n_components, available_components)

 
       self.pca_components[:used_components] = Vh[:used_components]
       if used_components < self.n_components:
          self.pca_components[used_components:] = 0.0  # padding with zeros

       self.mean[:] = mean
       self.fitted = True

    def forward(
        self,
        x: Float[Array, "batch channel height width"]
    ) -> Float[Array, "batch seq_len dim"]:
        x = x.view(x.size(0), -1)  # [batch, d_in]

        if not self.fitted:
        
            self._fit_pca_from_batch(x.detach())

        x_centered = x - self.mean  # [batch, d_in]
        x_pca = torch.matmul(x_centered, self.pca_components.T)  # [batch, n_components]
        out = self.linear(x_pca)  # [batch, d_out]
        return out.unsqueeze(1)  # [batch, 1, d_out]


## silu && swiglu

In [19]:
def silu(x: Float[Array,"... d_model"])->Float[Array,"... d_model"]:
    return x * torch.sigmoid(x)
class SwiGLU(nn.Module):
    def __init__(self,d_model:int,d_ff:int):
        super().__init__()
        self.w1= Linear(d_model, d_ff)
        self.w2 = Linear(d_ff, d_model)
        self.w3= Linear(d_model, d_ff)

    def forward(self,x:Float[Array,"... d_model"])->Float[Array,"... d_ff"]:

        return self.w2(silu(self.w1(x)* self.w3(x)))


## moe-like-topk-selector && universal classfier

In [20]:
class GeluGateProjector(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = Linear(d_model, d_ff)
        self.activation=nn.functional.gelu

    def forward(self, x: Float[Array, "... d_model"]) -> Float[Array, "... d_model"]:
        proj= self.w1(x)
        return self.activation(proj) * proj


class TopKSelector(nn.Module):
    def __init__(self, d_model:int,k:int):
        super().__init__()
        self.k = k
        self.score_net=nn.Sequential(
            Linear(d_model, d_model),
            SwiGLU(d_model, d_model)   
        ) 
        self.projector =GeluGateProjector(d_model,k)
    def forward(self,x:Float[Array,"... d_model"])->Float[Array,"... k"]:
        scores = self.score_net(x)
        topk_scores, topk_indices = torch.topk(scores, self.k, dim=-1)
        
        mask = torch.zeros_like(scores)
        mask.scatter_(dim=-1, index=topk_indices, value=1.0)
        x_masked = x * mask
        
        proj= self.projector(x_masked)
        return proj 
      

   

## universal classfier

In [21]:
class UniversalClassifier(nn.Module):
    def __init__(self, d_in:int,d_out:int):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in,d_out)
        )

    def forward(self, x: Float[Array, "batch seq_len dim"]) -> Float[Array, "batch num_classes"]:
        x = x.mean(dim=1)  
        return self.classifier(x)

# complete net class

In [22]:
class FeatureEncoder(nn.Module):
    def __init__(self, extractor: Extractor, selector: TopKSelector):
        super().__init__()
        self.extractor = extractor
        self.selector = selector

    def forward(self, x: Float[Array, "batch channel height width"]) -> Float[Array, "batch d_selected"]:
        feats = self.extractor(x)            # (B, Seq, D)
        selected = self.selector(feats)      # (B, k) or (B, d_selected)
        return selected

# helper func

In [23]:
MiB = 1024 ** 2

def model_size_b(model: nn.Module) -> int:
    size = 0
    for param in model.parameters():
        size += param.nelement() * param.element_size()
    for buf in model.buffers():
        size += buf.nelement() * buf.element_size()
    return size

def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def count_model_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [24]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold
from typing import Dict, List, Tuple, Union
import numpy as np
import pandas as pd

In [25]:
class ExperimentLogger:
    def __init__(self, model_name):
        self.model_name = model_name
        self.summary_records = []
        self.detail_records = []

    def log_summary(self, fold, epoch, train_acc, val_acc):
        self.summary_records.append({
            "Model": self.model_name,
            "Fold": fold,
            "Epoch": epoch,
            "TrainAcc": train_acc,
            "ValAcc": val_acc
        })

    def log_predictions(self, preds_batch):
        self.detail_records.extend(preds_batch)

    def save(self, summary_path, detail_path):
        pd.DataFrame(self.summary_records).to_csv(summary_path, index=False)
        pd.DataFrame(self.detail_records).to_csv(detail_path, index=False)

class ModelTrainer:
    def __init__(self, model, classifier, criterion, optimizer, device):
        self.model = model
        self.classifier = classifier
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device

    def train_epoch(self, loader):
        self.model.train()
        self.classifier.train()
        total, correct = 0, 0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            self.optimizer.zero_grad()
            feats = self.model(x)
            logits = self.classifier(feats)
            loss = self.criterion(logits, y)
            loss.backward()
            self.optimizer.step()
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
        return correct / total

    def eval_and_collect(self, loader, model_name, fold, epoch):
        self.model.eval()
        self.classifier.eval()
        total, correct = 0, 0
        batch_records = []
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                feats = self.model(x)
                logits = self.classifier(feats)
                preds = logits.argmax(dim=1)
                total += y.size(0)
                correct += (preds == y).sum().item()
                for i in range(len(x)):
                    batch_records.append({
                        "Model": model_name,
                        "Fold": fold,
                        "Epoch": epoch,
                        "true_label": y[i].item(),
                        "pred_label": preds[i].item(),
                        "correct": int(preds[i] == y[i])
                    })
        acc = correct / total
        return acc, batch_records


class ExperimentRunner:
    def __init__(self, models, dataset, num_folds, num_epochs, batch_size, lr, classfier_in):
        self.models = models
        self.dataset = dataset
        self.num_folds = num_folds
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.classfier_in = classfier_in
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.criterion = nn.CrossEntropyLoss()
        set_seed(42)

    def run(self):
        kf = KFold(n_splits=self.num_folds, shuffle=True, random_state=42)

        for name, model_instance in self.models.items():
            print(f"\\n>>> Model: {name}", count_model_params(model_instance))
            logger = ExperimentLogger(model_name=name)

            for fold, (train_idx, val_idx) in enumerate(kf.split(self.dataset)):
                train_loader = DataLoader(Subset(self.dataset, train_idx), batch_size=self.batch_size, shuffle=True)
                val_loader = DataLoader(Subset(self.dataset, val_idx), batch_size=self.batch_size, shuffle=False)

                model = copy.deepcopy(model_instance).to(self.device)
                classifier = UniversalClassifier(self.classfier_in, d_out=10).to(self.device) # Added d_out=10
                optimizer = torch.optim.Adam(list(model.parameters()) + list(classifier.parameters()), lr=self.lr)

                if isinstance(model, PCAMLP):
                    model.fit_pca(Subset(self.dataset, train_idx), self.device)

                trainer = ModelTrainer(model, classifier, self.criterion, optimizer, self.device)

                for epoch in range(1, self.num_epochs + 1):
                    train_acc = trainer.train_epoch(train_loader)
                    val_acc, preds = trainer.eval_and_collect(val_loader, name, fold, epoch)
                    logger.log_summary(fold, epoch, train_acc, val_acc)
                    logger.log_predictions(preds)
                    print(f"{name} Fold {fold} Epoch {epoch}: Train={train_acc:.4f}, Val={val_acc:.4f}")

            logger.save(f"{name}_summary.csv", f"{name}_details.csv")


# test

In [ ]:
models={
    
    "PCAMLP": FeatureEncoder(
        extractor=PCAMLP(d_in=32*32, d_out=16, d_hidden=64, img_size=32, n_components=256),
        selector=TopKSelector(d_model=16, k=8)),
     "MiniViT": FeatureEncoder(
        extractor=MiniVisionTransformer(d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),
        selector=TopKSelector(d_model=16, k=8)),
    "MiniUNet": FeatureEncoder(
        extractor=MiniUNet(d_in=1, d_out=12, d_hidden=28,kernel_num=3, img_size=32),
        selector=TopKSelector(d_model=12, k=8)),
   
    
    "MLP": FeatureEncoder(
        extractor=MLPExtractor(d_in=32*32, d_out=10, d_hidden=17, img_size=32),
        selector=TopKSelector(d_model=10, k=8)),    
  
    
}
for key,values in models.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

In [ ]:
runner = ExperimentRunner(
    models=models,
    dataset=full_dataset,
    num_folds=5,
    num_epochs=5,
    batch_size=64,
    lr=1e-3,
    classfier_in=8
)
runner.run()

In [ ]:


for name, feature_encoder in models.items():
    save_dict = {
        'extractor_class': type(feature_encoder.extractor).__name__,
        'extractor_init': {
            'd_in': feature_encoder.extractor.d_in,
            'd_out': feature_encoder.extractor.d_out,
            'd_hidden': getattr(feature_encoder.extractor, 'd_hidden', None),
            'patch_size': getattr(feature_encoder.extractor, 'patch_size', None),
            'num_heads': getattr(feature_encoder.extractor, 'num_heads', None),
            'img_size': getattr(feature_encoder.extractor, 'img_size', None),
            'n_components': getattr(feature_encoder.extractor, 'n_components', None),
            'kernel_num': getattr(feature_encoder.extractor, 'kernel_num', None),
        },
        'extractor_state_dict': feature_encoder.extractor.state_dict(),

        'selector_class': type(feature_encoder.selector).__name__,
        'selector_init': {
            'd_model': feature_encoder.extractor.d_out,
            'k': feature_encoder.selector.k,
        },
        'selector_state_dict': feature_encoder.selector.state_dict()
    }

 
    save_dict['extractor_init'] = {
        k: v for k, v in save_dict['extractor_init'].items() if v is not None
    }

  
    torch.save(save_dict, f'{name}_checkpoint.pth')

In [ ]:
def measure_inference_time(model: nn.Module, input_tensor: torch.Tensor, num_runs: int = 100) -> float:
    import time
    model.eval()
    with torch.no_grad():
        for _ in range(10):  # warmup
            _ = model(input_tensor)
        
        if torch.backends.mps.is_available():
            torch.mps.synchronize()

        start_time = time.perf_counter()
        for _ in range(num_runs):
            _ = model(input_tensor)
        if torch.backends.mps.is_available():
            torch.mps.synchronize()

        end_time = time.perf_counter()
        return (end_time - start_time) / num_runs * 1000  # ms


In [ ]:
for key, model in models.items():
    input_tensor = torch.randn(1, 1, 32, 32)  # Example input tensor
    print(key)
    inference_time = measure_inference_time(model, input_tensor)
    print(f"{key} average inference time: {inference_time:.4f} ms")

# flow mathching

## sample gauss

In [26]:
class Sampleable(ABC):

    @abstractmethod
    def sample(self, num_samples: int) -> Tuple[Float[Array, "batch ..."], Optional[Float[Array, "batch label_dim"]]]:
        pass
   
class IsotropicGaussian(nn.Module, Sampleable):
    def __init__(self, shape: List[int], std: float = 1.0):

        super().__init__()
        self.shape = shape
        self.std = std
        self.dummy = nn.Buffer(torch.zeros(1)) 
        
    def sample(self, num_samples:int) -> Tuple[Float[Array, "num_samples *shape"], Optional[None]]:
        return self.std * torch.randn(num_samples, *self.shape).to(self.dummy.device), None
    



## scalar

In [27]:
class Alpha(ABC):
    def __init__(self):
        # Check alpha_t(0) = 0
        assert torch.allclose(
            self(torch.zeros(1,1,1,1)), torch.zeros(1,1,1,1)
        )
        # Check alpha_1 = 1
        assert torch.allclose(
            self(torch.ones(1,1,1,1)), torch.ones(1,1,1,1)
        )
        
    @abstractmethod
    def __call__(self, t: Float[Array,'num_samples 1 1 1 ']) -> Float[Array, 'num_samples 1 1 1']:
        """
        Evaluates alpha_t. Should satisfy: self(0.0) = 0.0, self(1.0) = 1.0.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - alpha_t (num_samples, 1, 1, 1)
        """ 
        pass

    def dt(self, t: Float[Array,'num_samples  1 1 1']) -> Float[Array, 'num_samples 1 1 1']:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - d/dt alpha_t (num_samples, 1, 1, 1)
        """ 
        #t = t.unsqueeze(1)
        t= rearrange(t, 'b 1 1 1 -> b 1 1 1')
        dt = vmap(jacrev(self))(t)
        #dt.view(-1, 1, 1, 1)
        dt= rearrange(dt, 'b ... -> b 1 1 1')
        return dt
    

class LinearAlpha(Alpha):
    
    def __call__(self, t:  Float[Array,'num_samples 1 1 1']) ->  Float[Array,'num_samples 1 1 1']:
        return t

    def dt(self, t:  Float[Array,'num_samples 1 1 1']) ->  Float[Array,'num_samples 1 1 1']:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - d/dt alpha_t (num_samples, 1, 1, 1)
        """ 
        return torch.ones_like(t)
    

class Beta(ABC):
    def __init__(self):
        # Check beta_0 = 1
        assert torch.allclose(
            self(torch.zeros(1,1,1,1)), torch.ones(1,1,1,1)
        )
        # Check beta_1 = 0
        assert torch.allclose(
            self(torch.ones(1,1,1,1)), torch.zeros(1,1,1,1)
        )
        
    @abstractmethod
    def __call__(self, t: Float[Array,'num_samples 1 1 1']) -> Float[Array, 'num_samples 1 1 1']:
        """
        Evaluates alpha_t. Should satisfy: self(0.0) = 1.0, self(1.0) = 0.0.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - beta_t (num_samples, 1, 1, 1)
        """ 
        pass 

    def dt(self, t: Float[Array,'num_samples 1 1 1']) -> Float[Array,'num_samples 1 1 1']:
        """
        Evaluates d/dt beta_t.
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - d/dt beta_t (num_samples, 1, 1, 1)
        """ 
        t= rearrange(t, 'b 1 1 1 -> b 1 1 1')
        dt = vmap(jacrev(self))(t)
        #dt.view(-1, 1, 1, 1)
        dt= rearrange(dt, 'b ... -> b 1 1 1')
        return dt
    
class LinearBeta(Beta):
    def __call__(self, t: Float[Array,'num_samples 1 1 1']) -> Float[Array,'num_samples 1 1 1']:

        return 1-t
        
    def dt(self, t: Float[Array,'num_samples 1 1 1']) -> Float[Array,'num_samples 1 1 1']:
        return - torch.ones_like(t)
    

##  probpath

In [28]:
class ConditionalProbabilityPath(nn.Module, ABC):
    def __init__(self, p_simple: Sampleable, p_data: Sampleable):
        super().__init__()
        self.p_simple = p_simple
        self.p_data = p_data

    def sample_marginal_path(self, t: Float[Array,"num_sample 1 1 1"]) -> Float[Array, "num_sample c h w"]:
        """
        Samples from the marginal distribution p_t(x) = p_t(x|z) p(z)
        Args:
            - t: time (num_samples, 1, 1, 1)
        Returns:
            - x: samples from p_t(x), (num_samples, c, h, w)
        """
        num_samples = t.shape[0]
        # Sample conditioning variable z ~ p(z)
        z, _ = self.sample_conditioning_variable(num_samples) # (num_samples, c, h, w)
        # Sample conditional probability path x ~ p_t(x|z)
        x = self.sample_conditional_path(z, t) # (num_samples, c, h, w)
        return x
    @abstractmethod
    def sample_conditioning_variable(self, num_samples: int) -> Tuple[
        Float[Array, "num_samples c h w"],  # z
        Float[Array, "num_samples label_dim"]  # y
    ]:
        """
        Samples the conditioning variable z and label y
        """
        pass

    @abstractmethod
    def sample_conditional_path(
        self,
        z: Float[Array, "num_samples c h w"],
        t: Float[Array, "num_samples 1 1 1"]
    ) -> Float[Array, "num_samples c h w"]:
        """
        Samples from the conditional distribution p_t(x|z)
        """
        pass

    @abstractmethod
    def conditional_vector_field(
        self,
        x: Float[Array, "num_samples c h w"],
        z: Float[Array, "num_samples c h w"],
        t: Float[Array, "num_samples 1 1 1"]
    ) -> Float[Array, "num_samples c h w"]:
        """
        Evaluates the conditional vector field u_t(x|z)
        """
        pass

    @abstractmethod
    def conditional_score(
        self,
        x: Float[Array, "num_samples c h w"],
        z: Float[Array, "num_samples c h w"],
        t: Float[Array, "num_samples 1 1 1"]
    ) -> Float[Array, "num_samples c h w"]:
        """
        Evaluates the conditional score of p_t(x|z)
        """
        pass
    
    
    
    
    
class GaussianConditionalProbabilityPath(ConditionalProbabilityPath):
    def __init__(self, p_data: Sampleable, p_simple_shape: List[int], alpha: Alpha, beta: Beta):
        p_simple = IsotropicGaussian(shape = p_simple_shape, std=1.0)
        super().__init__(p_simple, p_data)
        self.alpha = alpha
        self.beta = beta
        
    def sample_conditioning_variable(self, num_samples: int) -> Tuple[
        Float[Array, "num_samples c h w"],  # z
        Float[Array, "num_samples label_dim"]  ]:# y
        return self.p_data.sample(num_samples)
    
    def sample_conditional_path(self, z: Float[Array,"num_samples c  h w"], t: Float[Array,"num_samples  1  1  1"]) -> Float[
        Array,"num_samples  c  h  w"]:
    
        return self.alpha(t) * z + self.beta(t) * torch.randn_like(z)
    
    def conditional_vector_field(self, 
                                 x: Float[Array,"num_samples  c  h w"], 
                                 z:Float[Array,"num_samples  c h w"], 
                                 t: Float[Array,"num_samples  1 1  1"]) -> Float[Array,"num_samples  c  h w"]:
    
        alpha_t = self.alpha(t) # (num_samples, 1, 1, 1)
        beta_t = self.beta(t) # (num_samples, 1, 1, 1)
        dt_alpha_t = self.alpha.dt(t) # (num_samples, 1, 1, 1)
        dt_beta_t = self.beta.dt(t) # (num_samples, 1, 1, 1)

        return (dt_alpha_t - dt_beta_t / beta_t * alpha_t) * z + dt_beta_t / beta_t * x
    
    
    def conditional_score(self, 
                            x: Float[Array,"num_samples  c h  w"], 
                                 z:Float[Array,"num_samples  c  h w"], 
                                 t: Float[Array,"num_samples 1  1  1"]) -> Float[Array,"num_samples  c  h  w"]:
        alpha_t = self.alpha(t)
        beta_t = self.beta(t)
        return (z * alpha_t - x) / beta_t ** 2 + 1e-4
    

## simlator

In [29]:
class ODE(ABC):
    @abstractmethod
    def drift_coefficient(
        self,
        xt: Float[Array, "bs c h w"],
        t: Float[Array, "bs 1"],
        **kwargs
    ) -> Float[Array, "bs c h w"]:
     
        pass

class SDE(ABC):
    @abstractmethod
    def drift_coefficient(
        self,
        xt: Float[Array, "bs c h w"],
        t: Float[Array, "bs 1 1 1"],
        **kwargs
    ) -> Float[Array, "bs c h w"]:
    
        pass

    @abstractmethod
    def diffusion_coefficient(
        self,
        xt: Float[Array, "bs c h w"],
        t: Float[Array, "bs 1 1 1"],
        **kwargs
    ) -> Float[Array, "bs c h w"]:
    
        pass
    

class Simulator(ABC):
    @abstractmethod
    def step(
        self, 
        xt: Float[Array, "bs c h w"],
        t: Float[Array, "bs 1 1 1"],
        dt: Float[Array, "bs 1 1 1"],
        **kwargs)-> Float[Array, "bs c h w"]:
        pass

    @torch.no_grad()
    def simulate(self, 
                 x: Float[Array, "bs c h w"],
                 ts: Float[Array, "bs nts 1 1 1"], 
                 **kwargs)-> Float[Array, "bs c h w"]:
        nts = ts.shape[1]
        for t_idx in tqdm(range(nts - 1)):
            t = ts[:, t_idx]
            h = ts[:, t_idx + 1] - ts[:, t_idx]
            x = self.step(x, t, h, **kwargs)
        return x
    
    @torch.no_grad()
    def simulate_with_trajectory(self, 
                                 x: Float[Array, "bs c h w"], 
                                 ts: Float[Array, "bs nts 1 1 1"], 
                                 **kwargs)-> Float[Array, "bs nts c h w"]:
        xs = [x.clone()]
        nts = ts.shape[1]
        for t_idx in tqdm(range(nts - 1)):
            t = ts[:,t_idx]
            h = ts[:, t_idx + 1] - ts[:, t_idx]
            x = self.step(x, t, h, **kwargs)
            xs.append(x.clone())
        return torch.stack(xs, dim=1)

    
class EulerSimulator(Simulator):
    def __init__(self, ode: ODE):
        self.ode = ode
        
    def step(self, 
             xt: torch.Tensor, 
             t: torch.Tensor,
             h: torch.Tensor, **kwargs):
        return xt + self.ode.drift_coefficient(xt,t, **kwargs) * h
    
class EulerMaruyamaSimulator(Simulator):
    def __init__(self, sde: SDE):
        self.sde = sde
        
    def step(self, xt: torch.Tensor, t: torch.Tensor, h: torch.Tensor, **kwargs):
        return xt + self.sde.drift_coefficient(xt,t, **kwargs) * h + self.sde.diffusion_coefficient(xt,t, **kwargs) * torch.sqrt(h) * torch.randn_like(xt)

In [30]:
class ConditionalVectorField(nn.Module, ABC):
    """
    MLP-parameterization of the learned vector field u_t^theta(x)
    """

    @abstractmethod
    def forward(self, x: Float[Array,"bs c h w"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array, "..."]:
        """
        Args:
        - x: (bs, c, h, w)
        - t: (bs, 1, 1, 1)
        - y: (bs,)
        Returns:
        - u_t^theta(x|y): (bs, c, h, w)
        """
        pass

class CFGVectorFieldODE(ODE):
    def __init__(self, net: ConditionalVectorField, guidance_scale: float = 1.0):
        self.net = net
        self.guidance_scale = guidance_scale

    def drift_coefficient(self, x: Float[Array,"bs c h w"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) ->Float[Array, "..."]:
        """
        Args:
        - x: (bs, c, h, w)
        - t: (bs, 1, 1, 1)
        - y: (bs,)
        """
        guided_vector_field = self.net(x, t, y)
        unguided_y = torch.ones_like(y) * 10
        unguided_vector_field = self.net(x, t, unguided_y)
        return (1 - self.guidance_scale) * unguided_vector_field + self.guidance_scale * guided_vector_field

## mnist sampler

In [31]:
class MNISTSampler(nn.Module, Sampleable):
    def __init__(self):
        super().__init__()
        self.dataset = datasets.MNIST(
            root="./data",
            train=True,
            download=False,
            transform=transforms.Compose([
                transforms.Resize((32, 32)),
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,)),
            ])
        )
        self.dummy = nn.Buffer(torch.zeros(1)) # Will automatically be moved when self.to(...) is called...

    def sample(self, num_samples: int) -> Tuple[Float[Array, "batch ..."], Optional[Float[Array, "batch label_dim"]]]:
        if num_samples > len(self.dataset):
            raise ValueError(f"num_samples exceeds dataset size: {len(self.dataset)}")
        
        indices = torch.randperm(len(self.dataset))[:num_samples]
        samples, labels = zip(*[self.dataset[i] for i in indices])
        samples = torch.stack(samples).to(self.dummy)
        labels = torch.tensor(labels, dtype=torch.int64).to(self.dummy.device)
        return samples, labels

## matcher

In [32]:
class FourierEncoder(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        self.half_dim = dim // 2
        self.weights = nn.Parameter(torch.randn(1, self.half_dim))

    def forward(self, t: Float[Array,'bs 1 1 1']) -> Float[Array,'bs dim']:
        t = t.view(-1, 1) # (bs, 1)
        freqs = t * self.weights * 2 * math.pi # (bs, half_dim)
        sin_embed = torch.sin(freqs) # (bs, half_dim)
        cos_embed = torch.cos(freqs) # (bs, half_dim)
        return torch.cat([sin_embed, cos_embed], dim=-1) * math.sqrt(2) # (bs, dim

In [33]:
class Matcher(nn.Module):
    def __init__(self, d_in:Int,latent_model: FeatureEncoder,out_channels:int,img_size:int=32 ):
        
        super().__init__()
        self.t_embeder= FourierEncoder(dim=d_in)
        self.y_embedder = Embedding(num_embeddings=11, embedding_dim=d_in)
        
        self.init_conv = nn.ConvTranspose2d(
    in_channels=d_in, out_channels=64, kernel_size=4, stride=2
)
        self.docoder=nn.Sequential(
            nn.Upsample(size=(img_size, img_size), mode='bilinear', align_corners=False),
            nn.GELU(), 
            Conv2d(64, 8, kernel_size=1, stride=1), 
            nn.BatchNorm2d(8) ,
         
                
            Conv2d(8, out_channels, kernel_size=1, stride=1),
            #nn.GELU(),
            #nn.BatchNorm2d(out_channels) 
        )
        
        self.latent_model = latent_model
        
        
        
    def forward(self, x: Float[Array,"bs 1 32 32"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array,"bs 1 32 32"]:
        t_embed = self.t_embeder(t)
        y_embed = self.y_embedder(y)
        latent = self.latent_model(x)
        x_y=einsum(latent,y_embed,'b seq d ,b d-> b seq d')
        x_t=einsum(x_y, t_embed, 'b seq d , b d -> b seq d ')
        
        x_t = rearrange(x_t, 'b seq d -> b d seq 1')
        x_t = self.init_conv(x_t)
       
        x_t = self.docoder(x_t)
    
        return x_t

## vec filed net

In [34]:
class VecField(nn.Module):
    def __init__(self,matcher:Matcher):
        super().__init__()
        self.matcher=matcher
        
    def forward(self, x: Float[Array,"bs c h w"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array, "bs c h w"]:
        x=self.matcher(x,t,y)
        return x
        

## flow loss

In [35]:
class CFG(nn.Module):
    def __init__(self,path: GaussianConditionalProbabilityPath, model: VecField, eta: float,):
      super().__init__()
      self.eta=eta
      self.path=path
      self.model=model
    
    def forward( self,batch_size: int)->Float[Array,'...']:
        z, y = self.path.p_data.sample(batch_size)
        xi = torch.rand(y.shape[0]).to(y.device)
        y[xi < self.eta] = 10.0
        
        t = torch.rand(batch_size,1,1,1).to(z) # (bs, 1, 1, 1)
        x = self.path.sample_conditional_path(z,t) # (bs, 1, 32, 32)
        
        
        ut_theta = self.model(x,t,y) # (bs, 1, 32, 32)
        ut_ref = self.path.conditional_vector_field(x,z,t) # (bs, 1, 32, 32)
        error = torch.einsum('bchw -> b', torch.square(ut_theta - ut_ref)) # (bs,)
        return torch.mean(error)
        
    

## flow trainer

In [36]:
class FlowLogger:
    def __init__(self, model_name):
        self.model_name = model_name
        self.loss_records = []

    def log(self, epoch, train_loss, val_loss=None):
        self.loss_records.append({
            "Model": self.model_name,
            "Epoch": epoch,
            "TrainLoss": train_loss,
            "ValLoss": val_loss if val_loss is not None else -1
        })

    def save(self, path="loss_log.csv"):
        import pandas as pd
        df = pd.DataFrame(self.loss_records)
        df.to_csv(path, index=False)
        
        


In [37]:
class FlowModelTrainer:
 def __init__(self,device, model: nn.Module,loss_:CFG,logger=FlowLogger,):
        super().__init__()
        self.model = model
        self.get_loss=loss_
        self.logger = logger
 def get_optimizer(self, lr: float):
        return torch.optim.Adam(self.model.parameters(), lr=lr)
 
 def train(self,  num_epochs: int, batch_size: int, device=device, lr: float = 1e-3) ->Float[Array, ""]:
        self.model.to(device)
        opt = self.get_optimizer(lr)
        self.model.train()
        
        pbar = tqdm(enumerate(range(num_epochs)))
        for idx, epoch in pbar:
            opt.zero_grad()
            loss = self.get_loss(batch_size)
            loss.backward()
            opt.step()
            pbar.set_description(f'Epoch {idx}, loss: {loss.item():.3f}')
            self.model.eval()
            self.get_loss.eval()
            with torch.no_grad():
                val_loss = self.get_loss(batch_size)

            if self.logger:
                self.logger.log(epoch,loss.item(), val_loss.item())

            pbar.set_postfix({
                "train": f"{loss.item():.4f}",
                "val": f"{val_loss.item():.4f}"
            })
        
       


In [38]:
class FlowExperimentRunner:
    def __init__(self, models, cfg_class, path, num_epochs: int, batch_size: int,
                 device, eta: float = 0.1, lr: float = 1e-3):
        self.models = models
        self.cfg_class = cfg_class
        self.path = path
        self.eta = eta
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device

    def run(self):
        for name, model in self.models.items():
            print(f"\n>>> Training Model: {name}")
            logger = FlowLogger(model_name=name)

            cfg_loss = self.cfg_class(path=self.path, model=model, eta=self.eta)
            trainer = FlowModelTrainer(model=model, loss_=cfg_loss,
                                       device=self.device, logger=logger)

            trainer.train(num_epochs=self.num_epochs,
                          batch_size=self.batch_size,
                          lr=self.lr)

            logger.save(f"{name}_flow_loss.csv")

## forward-inverse path 

In [39]:
class LatentCFG(nn.Module):
    def __init__(self,forward_model,inverse_model,path: GaussianConditionalProbabilityPath, eta: float,):
      super().__init__()
      self.eta=eta
      self.path=path
      self.forward_model=forward_model
      self.inverse_model=inverse_model
    
    def forward( self,batch_size: int)->Float[Array,'...']:
        z, y = self.path.p_data.sample(batch_size)
        xi = torch.rand(y.shape[0]).to(y.device)
        y[xi < self.eta] = 10.0
        
        t = torch.rand(batch_size,1,1,1).to(z) # (bs, 1, 1, 1)
        x = self.path.sample_conditional_path(z,t) # (bs, 1, 32, 32)
        
        
        ut_theta_latent = self.forward_model(x,t,y) # (bs, seq,d)
        ut_ref = self.path.conditional_vector_field(x,z,t) # (bs, 1, 32, 32)
        ut_ref_latent=self.inverse_model(ut_ref)# (bs, seq,d)
        error = einsum(torch.square( ut_theta_latent-ut_ref_latent),'b seq d -> b')
        return torch.mean(error)

''''
class InverseCFG(nn.Module):
    def __init__(self,decoder_model,inverse_model,path: GaussianConditionalProbabilityPath, eta: float,):
      super().__init__()
      self.eta=eta
      self.path=path
      self.decoder_model=decoder_model
      self.inverse_model=inverse_model
    
    def forward( self,batch_size: int)->Float[Array,'...']:
        z, y = self.path.p_data.sample(batch_size)
        xi = torch.rand(y.shape[0]).to(y.device)
        y[xi < self.eta] = 10.0
        
        t = torch.rand(batch_size,1,1,1).to(z) # (bs, 1, 1, 1)
        x = self.path.sample_conditional_path(z,t) # (bs, 1, 32, 32)
        
        ut_ref = self.path.conditional_vector_field(x,z,t) # (bs, 1, 32, 32)
        latent=self.inverse_model(ut_ref)
        ut_decode=self.decoder_model(latent)
        error = einsum(torch.square( ut_decode- ut_ref),'b c h w -> b')
        return torch.mean(error)'''

class InverseutCFG(nn.Module):
    def __init__(self,decoder_model,inverse_model,path: GaussianConditionalProbabilityPath, eta: float,):
      super().__init__()
      self.eta=eta
      self.path=path
      self.decoder_model=decoder_model
      self.inverse_model=inverse_model
    
    def forward( self,batch_size: int)->Float[Array,'...']:
        z, y = self.path.p_data.sample(batch_size)
        xi = torch.rand(y.shape[0]).to(y.device)
        y[xi < self.eta] = 10.0
        
        t = torch.rand(batch_size,1,1,1).to(z) # (bs, 1, 1, 1)
        x = self.path.sample_conditional_path(z,t) # (bs, 1, 32, 32)
        
        ut_ref = self.path.conditional_vector_field(x,z,t) # (bs, 1, 32, 32)
        latent=self.inverse_model(ut_ref)
        ut_decode=self.decoder_model(latent,t=t,y=y)
        error = einsum(torch.square( ut_decode- ut_ref),'b c h w -> b')
        return torch.mean(error)
      


In [40]:
class contexter(nn.Module):
     def __init__(self,latent_model,d_in:Int):
        super().__init__()
        self.t_embeder= FourierEncoder(dim=d_in)
        self.y_embedder = Embedding(num_embeddings=11, embedding_dim=d_in)
        self.latent_model=latent_model
    
     def forward(self, x: Float[Array,"bs 1 32 32"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array,"b seq d"]:
        t_embed = self.t_embeder(t)
        y_embed = self.y_embedder(y)
        latent_output = self.latent_model(x)
        x_y=einsum(latent_output,y_embed,'b seq d ,b d-> b seq d')
        x_t=einsum(x_y, t_embed, 'b seq d , b d -> b seq d ')
        return x_t
class LatentVecField(nn.Module):
    def __init__(self,latent,d_in:Int):
        super().__init__()
        self.contexter=contexter(latent,d_in)
        
    def forward(self, x: Float[Array,"bs c h w"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array, "bs seq  d"]:
        x=self.contexter(x,t,y)
        return x
    
class vecfield(nn.Module):
    def __init__(self,encoder,decoder):
        super().__init__()
        self.encoder=encoder
        self.decoder=decoder
    
    
    def forward(self, x: Float[Array,"bs 1 32 32"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array,"bs 1 32 32"]:
        latent=self.encoder(x,t=t,y=y)
        return self.decoder(latent,t=t,y=y)

class Utdecoder(nn.Module):
    def __init__ (self, d_in: int, d_out: int, img_size=32):
        super().__init__()
        self.init_conv = nn.ConvTranspose2d(in_channels=d_in, out_channels=64, kernel_size=4, stride=2)
        self.t_embeder= FourierEncoder(dim=d_in)
        self.y_embedder = Embedding(num_embeddings=11, embedding_dim=d_in)
        self.decoder = nn.Sequential(
            nn.Upsample(size=(img_size, img_size), mode='bilinear', align_corners=False),
            nn.GELU(),
            Conv2d(64, 8, kernel_size=1, stride=1),
            nn.BatchNorm2d(8),
            Conv2d(8, out_channels=d_out, kernel_size=1, stride=1),
            #nn.GELU(),
            #nn.BatchNorm2d(d_out)
        )

    def forward(self, x: Float[Array,'b seq d'], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array,"bs c h w"]:
        t_embed = self.t_embeder(t)
        y_embed = self.y_embedder(y)
        x_y=einsum(x,y_embed,'b seq d ,b d-> b seq d')
        x_t=einsum(x_y, t_embed, 'b seq d , b d -> b seq d ')
        x = rearrange(x, 'b seq d -> b d seq 1')
        x = self.init_conv(x)
        x = self.decoder(x)
        return x

In [41]:
class LatentFlowModelTrainer2:
 def __init__(self,device, forward_model: nn.Module,inverse_model: nn.Module,loss_:LatentCFG,logger=FlowLogger,):
        super().__init__()
        self.forward_model = forward_model
        self.inverse_model=inverse_model
        self.get_loss=loss_
        self.logger = logger
 def get_optimizer(self, lr: float):
        return torch.optim.Adam(self.forward_model.parameters(), lr=lr)
 
 def train(self,  num_epochs: int, batch_size: int, device=device, lr: float = 1e-3) ->Float[Array, ""]:
        self.forward_model.to(device)
        self.inverse_model.to(device)
        forward_opt = self.get_optimizer(lr)
        self.forward_model.train()
        self.inverse_model.eval()
        
        pbar = tqdm(enumerate(range(num_epochs)))
        for idx, epoch in pbar:
            forward_opt.zero_grad()
           
            loss = self.get_loss(batch_size)
            loss.backward()
            forward_opt.step()
          
            
            pbar.set_description(f'Epoch {idx}, loss: {loss.item():.3f}')
            self.forward_model.eval()
            self.inverse_model.eval()
            self.get_loss.eval()
            with torch.no_grad():
                val_loss = self.get_loss(batch_size)

            if self.logger:
                self.logger.log(epoch,loss.item(), val_loss.item())

            pbar.set_postfix({
                "train": f"{loss.item():.4f}",
                "val": f"{val_loss.item():.4f}"
            })
            
class LatentFlowModelTrainer:
 def __init__(self,device, forward_model: nn.Module,inverse_model: nn.Module,loss_:LatentCFG,logger=FlowLogger,):
        super().__init__()
        self.forward_model = forward_model
        self.inverse_model=inverse_model
        self.get_loss=loss_
        self.logger = logger
        
        
        
 def get_optimizer(self, lr: float):
        return torch.optim.Adam(self.forward_model.parameters(), lr=lr),torch.optim.Adam(self.inverse_model.parameters(), lr=lr)
 
 def train(self,  num_epochs: int, batch_size: int, device=device, lr: float = 1e-3) ->Float[Array, ""]:
        self.forward_model.to(device)
        self.inverse_model.to(device)
        forward_opt,inverse_opt = self.get_optimizer(lr)
        self.forward_model.train()
        self.inverse_model.train()
        
        pbar = tqdm(enumerate(range(num_epochs)))
        for idx, epoch in pbar:
            forward_opt.zero_grad()
            inverse_opt.zero_grad()
            loss = self.get_loss(batch_size)
            loss.backward()
            forward_opt.step()
            inverse_opt.step()
            
            pbar.set_description(f'Epoch {idx}, loss: {loss.item():.3f}')
            self.forward_model.eval()
            self.inverse_model.eval()
            self.get_loss.eval()
            with torch.no_grad():
                val_loss = self.get_loss(batch_size)

            if self.logger:
                self.logger.log(epoch,loss.item(), val_loss.item())

            pbar.set_postfix({
                "train": f"{loss.item():.4f}",
                "val": f"{val_loss.item():.4f}"
            })
            
            


In [42]:

            
class LatentFlowExperimentRunner:
    def __init__(self,
                 models: dict,  # e.g. {"ViT": {"forward": ..., "inverse": ...}, ...}
                 cfg_class: LatentCFG,
                 path,
                 num_epochs: int,
                 batch_size: int,
                 device,
                 eta: float = 0.1,
                 lr: float = 1e-3):
        self.models = models
        self.cfg_class = cfg_class
        
        
        self.path = path
        self.eta = eta
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device

    def run(self):
        for name, model_pair in self.models.items():
            print(f"\n>>> Training Model: {name}")
            forward_model = model_pair["forward"]
            inverse_model = model_pair["inverse"]

            logger = FlowLogger(model_name=name)

           
            cfg_loss = self.cfg_class(
                forward_model=forward_model,
                inverse_model=inverse_model,
                path=self.path,
                eta=self.eta
            )

          
            trainer = LatentFlowModelTrainer(
                device=self.device,
                forward_model=forward_model,
                inverse_model=inverse_model,
                loss_=cfg_loss,
                logger=logger
            )

        
            trainer.train(
                num_epochs=self.num_epochs,
                batch_size=self.batch_size,
                lr=self.lr
            )

            logger.save(f"{name}_flow_loss.csv")
            

            
class LatentFlowExperimentRunner2:
    def __init__(self,
                 models: dict,  # e.g. {"ViT": {"forward": ..., "inverse": ...}, ...}
                 cfg_class: LatentCFG,
                 path,
                 num_epochs: int,
                 batch_size: int,
                 device,
                 eta: float = 0.1,
                 lr: float = 1e-3):
        self.models = models
        self.cfg_class = cfg_class
        self.path = path
        self.eta = eta
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device

    def run(self):
        for name, model_pair in self.models.items():
            print(f"\n>>> Training Model: {name}")
            forward_model = model_pair["forward"]
            inverse_model = model_pair["inverse"]

            logger = FlowLogger(model_name=name)

           
            cfg_loss = self.cfg_class(
                forward_model=forward_model,
                inverse_model=inverse_model,
                path=self.path,
                eta=self.eta
            )

          
            trainer = LatentFlowModelTrainer2(
                device=self.device,
                forward_model=forward_model,
                inverse_model=inverse_model,
                loss_=cfg_loss,
                logger=logger
            )

        
            trainer.train(
                num_epochs=self.num_epochs,
                batch_size=self.batch_size,
                lr=self.lr
            )

            logger.save(f"{name}_flow_loss.csv")
                        

In [43]:

class InverseFlowModelTrainer:
 def __init__(self,device, decoder_model: nn.Module,inverse_model: nn.Module,loss_:InverseutCFG,logger=FlowLogger,):
        super().__init__()
        self.decoder_model = decoder_model
        self.inverse_model=inverse_model
        self.get_loss=loss_
        self.logger = logger
 def get_optimizer(self, lr: float):
        return torch.optim.Adam(self.decoder_model.parameters(), lr=lr)
 
 def train(self,  num_epochs: int, batch_size: int, device=device, lr: float = 1e-3) ->Float[Array, ""]:
        self.decoder_model.to(device)
        self.inverse_model.to(device)
        opt = self.get_optimizer(lr)
        self.decoder_model.train()
        self.inverse_model.eval()
        
        pbar = tqdm(enumerate(range(num_epochs)))
        for idx, epoch in pbar:
            opt.zero_grad()
            loss = self.get_loss(batch_size)
            loss.backward()
            opt.step()
        
            pbar.set_description(f'Epoch {idx}, loss: {loss.item():.3f}')
            self.decoder_model.eval()
            self.inverse_model.eval()
            self.get_loss.eval()
            with torch.no_grad():
                val_loss = self.get_loss(batch_size)

            if self.logger:
                self.logger.log(epoch,loss.item(), val_loss.item())

            pbar.set_postfix({
                "train": f"{loss.item():.4f}",
                "val": f"{val_loss.item():.4f}"
            })
            
            
            

class InverseFlowModelTrainer2:
 def __init__(self,device, decoder_model: nn.Module,inverse_model: nn.Module,loss_:InverseutCFG,logger=FlowLogger,):
        super().__init__()
        self.decoder_model = decoder_model
        self.inverse_model=inverse_model
        self.get_loss=loss_
        self.logger = logger
 def get_optimizer(self, lr: float):
        return torch.optim.Adam(self.decoder_model.parameters(), lr=lr)
 
 def train(self,  num_epochs: int, batch_size: int, device=device, lr: float = 1e-3) ->Float[Array, ""]:
        self.decoder_model.to(device)
        self.inverse_model.to(device)
        opt = self.get_optimizer(lr)
        self.decoder_model.train()
        self.inverse_model.eval()
        
        pbar = tqdm(enumerate(range(num_epochs)))
        for idx, epoch in pbar:
            opt.zero_grad()
            loss = self.get_loss(batch_size)
            loss.backward()
            opt.step()
        
            pbar.set_description(f'Epoch {idx}, loss: {loss.item():.3f}')
            self.decoder_model.eval()
            self.inverse_model.eval()
            self.get_loss.eval()
            with torch.no_grad():
                val_loss = self.get_loss(batch_size)

            if self.logger:
                self.logger.log(epoch,loss.item(), val_loss.item())

            pbar.set_postfix({
                "train": f"{loss.item():.4f}",
                "val": f"{val_loss.item():.4f}"
            })

In [44]:

class InverseFlowExperimentRunner:
    def __init__(self,
                 models: dict,  
                 cfg_class: InverseutCFG,
                 path,
                 num_epochs: int,
                 batch_size: int,
                 device,
                 eta: float = 0.1,
                 lr: float = 1e-3):
        self.models = models
        self.cfg_class = cfg_class
        self.path = path
        self.eta = eta
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device

    def run(self):
        for name, model_pair in self.models.items():
            print(f"\n>>> Training Model: {name}")
            decoder_model = model_pair["decoder"]
            inverse_model = model_pair["inverse"]

            logger = FlowLogger(model_name=name)

           
            cfg_loss = self.cfg_class(
                decoder_model=decoder_model,
                inverse_model=inverse_model,
                path=self.path,
                eta=self.eta
            )

          
            trainer = InverseFlowModelTrainer(
                device=self.device,
                decoder_model=decoder_model,
                inverse_model=inverse_model,
                loss_=cfg_loss,
                logger=logger
            )

        
            trainer.train(
                num_epochs=self.num_epochs,
                batch_size=self.batch_size,
                lr=self.lr
            )

            logger.save(f"{name}_flow_loss.csv")


class InverseFlowExperimentRunner2:
    def __init__(self,
                 models: dict,  
                 cfg_class: InverseutCFG,
                 path,
                 num_epochs: int,
                 batch_size: int,
                 device,
                 eta: float = 0.1,
                 lr: float = 1e-3):
        self.models = models
        self.cfg_class = cfg_class
        self.path = path
        self.eta = eta
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device

    def run(self):
        for name, model_pair in self.models.items():
            print(f"\n>>> Training Model: {name}")
            decoder_model = model_pair["decoder"]
            inverse_model = model_pair["inverse"]

            logger = FlowLogger(model_name=name)

           
            cfg_loss = self.cfg_class(
                decoder_model=decoder_model,
                inverse_model=inverse_model,
                path=self.path,
                eta=self.eta
            )

          
            trainer = InverseFlowModelTrainer2(
                device=self.device,
                decoder_model=decoder_model,
                inverse_model=inverse_model,
                loss_=cfg_loss,
                logger=logger
            )

        
            trainer.train(
                num_epochs=self.num_epochs,
                batch_size=self.batch_size,
                lr=self.lr
            )

            logger.save(f"{name}_flow_loss.csv")

In [45]:
class biencoder(nn.Module):
    def __init__(self, encoder, invencoder):
        super().__init__()
        self.encoder = encoder
        self.invencoder = invencoder

    def forward(self, x: Float[Array, "bs c h w"], t: Float[Array, "bs 1 1 1"], y: Float[Array, "bs ..."]) -> Float[Array, "bs c h w"]:
        x = self.encoder(x, t, y)        
        x = self.invencoder(x)           
        return x
    
class FullCFG(nn.Module):
    def __init__(self,forward_model,inverse_model,decoder_model,path: GaussianConditionalProbabilityPath, eta: float,):
      super().__init__()
      self.eta=eta
      self.path=path
      self.forward_model=forward_model
      self.inverse_model=inverse_model
      self.decoder_model=decoder_model
    
    def forward( self,batch_size: int)->Float[Array,'...']:
        z, y = self.path.p_data.sample(batch_size)
        xi = torch.rand(y.shape[0]).to(y.device)
        y[xi < self.eta] = 10.0
        
        t = torch.rand(batch_size,1,1,1).to(z) 
        x = self.path.sample_conditional_path(z,t) 
        
        
        ut_theta_latent = self.forward_model(x,t,y) 
        ut_ref = self.path.conditional_vector_field(x,z,t)
        ut_ref_latent=self.inverse_model(ut_ref)
        ut_inverse_path=self.decoder_model(ut_ref_latent,t=t,y=y)
        ut_forward_path=self.decoder_model(ut_theta_latent,t=t,y=y)
        loss_latent=einsum(torch.square(ut_theta_latent-ut_ref_latent),'b seq d ->b').mean()
        loss_inverse_path=einsum(torch.square(ut_inverse_path- ut_ref),'b c h w ->b').mean()
        loss_forward_path=einsum(torch.square(ut_forward_path- ut_ref),'b c h w ->b').mean()
        loss_diff_path= einsum(torch.square(ut_forward_path-ut_inverse_path),'b c h w ->b').mean()
        #error = einsum(torch.square( ut_theta_latent-ut_ref_latent),'b seq d -> b')
        return ((loss_inverse_path+ loss_forward_path)/2+loss_latent+loss_diff_path)/3
    
    
class FullFlowModelTrainer:
 def __init__(self,device, forward_model: nn.Module,inverse_model: nn.Module,decoder_model:nn.Module,loss_:FullCFG,logger=FlowLogger,):
        super().__init__()
        self.forward_model = forward_model
        self.inverse_model=inverse_model
        self.decoder_model=decoder_model
        self.get_loss=loss_
        self.logger = logger
 def get_optimizer(self, lr: float):
        return (torch.optim.Adam(self.forward_model.parameters(), lr=lr),
    torch.optim.Adam(self.inverse_model.parameters(), lr=lr),
    torch.optim.Adam(self.inverse_model.parameters(), lr=lr))
                                
 
 def train(self,  num_epochs: int, batch_size: int, device=device, lr: float = 1e-3) ->Float[Array, ""]:
        self.forward_model.to(device)
        self.inverse_model.to(device)
        self.decoder_model.to(device)
        forward_opt,inverse_opt,decoder_opt = self.get_optimizer(lr)
        self.forward_model.train()
        self.inverse_model.train()
        self.decoder_model.train()
        
        pbar = tqdm(enumerate(range(num_epochs)))
        for idx, epoch in pbar:
            forward_opt.zero_grad()
            inverse_opt.zero_grad()
            decoder_opt.zero_grad()
            loss = self.get_loss(batch_size)
            loss.backward()
            forward_opt.step()
            inverse_opt.step()
            decoder_opt.step()
            
            pbar.set_description(f'Epoch {idx}, loss: {loss.item():.3f}')
            self.forward_model.eval()
            self.inverse_model.eval()
            self.decoder_model.eval()
            self.get_loss.eval()
            with torch.no_grad():
                val_loss = self.get_loss(batch_size)

            if self.logger:
                self.logger.log(epoch,loss.item(), val_loss.item())

            pbar.set_postfix({
                "train": f"{loss.item():.4f}",
                "val": f"{val_loss.item():.4f}"
            })
            
            
class FullFlowExperimentRunner:
    def __init__(self,
                 models: dict,  # e.g. {"ViT": {"forward": ..., "inverse": ...}, ...}
                 cfg_class: FullCFG,
                 path,
                 num_epochs: int,
                 batch_size: int,
                 device,
                 eta: float = 0.1,
                 lr: float = 1e-3):
        self.models = models
        self.cfg_class = cfg_class
        self.path = path
        self.eta = eta
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device

    def run(self):
        for name, model_pair in self.models.items():
            print(f"\n>>> Training Model: {name}")
            forward_model = model_pair["forward"]
            inverse_model = model_pair["inverse"]
            decoder_model= model_pair["decoder"]

            logger = FlowLogger(model_name=name)

           
            cfg_loss = self.cfg_class(
                forward_model=forward_model,
                inverse_model=inverse_model,
                decoder_model=decoder_model,
                path=self.path,
                eta=self.eta
            )

          
            trainer = FullFlowModelTrainer(
                device=self.device,
                forward_model=forward_model,
                inverse_model=inverse_model,
                decoder_model=decoder_model,
                loss_=cfg_loss,
                logger=logger
            )

        
            trainer.train(
                num_epochs=self.num_epochs,
                batch_size=self.batch_size,
                lr=self.lr
            )

            logger.save(f"{name}_flow_loss.csv")

In [46]:
'''class vecfield(nn.Module):
    def __init__(self,encoder,decoder):
        super().__init__()
        self.encoder=encoder
        self.decoder=decoder
    
    
    def forward(self, x: Float[Array,"bs 1 32 32"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array,"bs 1 32 32"]:
        latent=self.encoder(x,t=t,y=y)
        return self.decoder(latent,t=t,y=y)
class Utdecoder(nn.Module):
    def __init__ (self, d_in: int, d_out: int, img_size=32):
        super().__init__()
        self.init_conv = nn.ConvTranspose2d(in_channels=d_in, out_channels=64, kernel_size=4, stride=2)
        self.t_embeder= FourierEncoder(dim=d_in)
        self.y_embedder = Embedding(num_embeddings=11, embedding_dim=d_in)
        self.decoder = nn.Sequential(
            nn.Upsample(size=(img_size, img_size), mode='bilinear', align_corners=False),
            nn.GELU(),
            Conv2d(64, 8, kernel_size=1, stride=1),
            nn.BatchNorm2d(8),
            Conv2d(8, out_channels=d_out, kernel_size=1, stride=1),
            nn.GELU(),
            nn.BatchNorm2d(d_out)
        )

    def forward(self, x: Float[Array,'b seq d'], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array,"bs c h w"]:
        t_embed = self.t_embeder(t)
        y_embed = self.y_embedder(y)
        x_y=einsum(x,y_embed,'b seq d ,b d-> b seq d')
        x_t=einsum(x_y, t_embed, 'b seq d , b d -> b seq d ')
        x = rearrange(x, 'b seq d -> b d seq 1')
        x = self.init_conv(x)
        x = self.decoder(x)
        return x'''

'class vecfield(nn.Module):\n    def __init__(self,encoder,decoder):\n        super().__init__()\n        self.encoder=encoder\n        self.decoder=decoder\n    \n    \n    def forward(self, x: Float[Array,"bs 1 32 32"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array,"bs 1 32 32"]:\n        latent=self.encoder(x,t=t,y=y)\n        return self.decoder(latent,t=t,y=y)\nclass Utdecoder(nn.Module):\n    def __init__ (self, d_in: int, d_out: int, img_size=32):\n        super().__init__()\n        self.init_conv = nn.ConvTranspose2d(in_channels=d_in, out_channels=64, kernel_size=4, stride=2)\n        self.t_embeder= FourierEncoder(dim=d_in)\n        self.y_embedder = Embedding(num_embeddings=11, embedding_dim=d_in)\n        self.decoder = nn.Sequential(\n            nn.Upsample(size=(img_size, img_size), mode=\'bilinear\', align_corners=False),\n            nn.GELU(),\n            Conv2d(64, 8, kernel_size=1, stride=1),\n            nn.BatchNorm2d(8),\n            Conv2

# save&load

In [47]:
import os
def save_complex_models(nested_dict, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)

    for top_key, sub_dict in nested_dict.items():
        model_folder = os.path.join(save_dir, top_key)
        os.makedirs(model_folder, exist_ok=True)

        if isinstance(sub_dict, dict):
            for name, model in sub_dict.items():
               
                if isinstance(model, dict):
                    for subname, submodel in model.items():
                        path = os.path.join(model_folder, f"{name}_{subname}.pth")
                        print(f"Saving {top_key} / {name}_{subname} to {path}")
                        torch.save(submodel.state_dict(), path)
                else:
                    path = os.path.join(model_folder, f"{name}.pth")
                    print(f"Saving {top_key} / {name} to {path}")
                    torch.save(model.state_dict(), path)
        else:
          
            path = os.path.join(save_dir, f"{top_key}.pth")
            print(f"Saving {top_key} to {path}")
            torch.save(sub_dict.state_dict(), path)

    print(f"✔️ Models saved to {save_dir}")

def load_all_model_families(model_families, save_dir="saved_vecfield_models"):

    loaded_families = {}

    for family_name, model_dict in model_families.items():
        folder_path = os.path.join(save_dir, family_name)
        if not os.path.exists(folder_path):
            print(f"❌ folder {folder_path} not found, skipping.")
            continue

        loaded_families[family_name] = {}

        for model_name, model_obj in model_dict.items():
            if isinstance(model_obj, dict):
                loaded_families[family_name][model_name] = {}
                for subname, submodel in model_obj.items():
                    filename = f"{model_name}_{subname}.pth"
                    path = os.path.join(folder_path, filename)
                    if os.path.exists(path):
                        submodel.load_state_dict(torch.load(path, map_location='cpu'))
                        submodel.eval()
                        loaded_families[family_name][model_name][subname] = submodel
                        print(f"✅ Loaded {family_name}/{model_name}_{subname}")
                    else:
                        print(f"❌ Missing: {path}")
            else:
                filename = f"{model_name}.pth"
                path = os.path.join(folder_path, filename)
                if os.path.exists(path):
                    model_obj.load_state_dict(torch.load(path, map_location='cpu'))
                    model_obj.eval()
                    loaded_families[family_name][model_name] = model_obj
                    print(f"✅ Loaded {family_name}/{model_name}")
                else:
                    print(f"❌ Missing: {path}")

    return loaded_families

# test

In [48]:
path = GaussianConditionalProbabilityPath(
    p_data = MNISTSampler(),
    p_simple_shape = [1, 32, 32],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

In [49]:
device

device(type='mps')

## expr-origin

In [100]:
modelo = {
    "MiniViT": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32
                ),
                selector=TopKSelector(d_model=16, k=8)
            ),
            out_channels=1
        )
    ),
    "MiniUNet": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
                selector=TopKSelector(d_model=16, k=8)  #
            ),
            out_channels=1
        )
    ),
    "PCAMLP": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ),
                selector=TopKSelector(d_model=16, k=8)
            ),
            out_channels=1
        )
    ),
    "MLP": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
                selector=TopKSelector(d_model=14, k=8)
            ),
            out_channels=1
        )
    ),
}

In [101]:
for key,values in modelo.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

MiniViT model size: 0.10 MiB
MiniViT model params: 25941
MiniUNet model size: 0.10 MiB
MiniUNet model params: 26513
PCAMLP model size: 1.10 MiB
PCAMLP model params: 25694
MLP model size: 0.10 MiB
MLP model params: 26491


In [102]:
runner = FlowExperimentRunner(
    models=modelo,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


0it [00:00, ?it/s]

Epoch 2499, loss: 1185.204: : 2500it [05:56,  7.01it/s, train=1185.2039, val=1179.9536]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1213.310: : 2500it [08:58,  4.65it/s, train=1213.3096, val=1201.0107]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1184.703: : 2500it [03:26, 12.13it/s, train=1184.7026, val=1191.6694]



>>> Training Model: MLP


Epoch 2499, loss: 1204.559: : 2500it [03:25, 12.17it/s, train=1204.5588, val=1211.1820]


In [103]:
save_complex_models({"modelo": modelo }, save_dir="saved_models-o")

Saving modelo / MiniViT to saved_models-o/modelo/MiniViT.pth
Saving modelo / MiniUNet to saved_models-o/modelo/MiniUNet.pth
Saving modelo / PCAMLP to saved_models-o/modelo/PCAMLP.pth
Saving modelo / MLP to saved_models-o/modelo/MLP.pth
✔️ Models saved to saved_models-o


## expr-1 forward path end2end

In [96]:
models_full={ "MiniViT":
    vecfield( LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),Utdecoder(16,1)),
    "MiniUNet":
    vecfield( LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),Utdecoder(16,1)),
    "PCAMLP":
    vecfield(LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),Utdecoder(16,1)),
    "MLP":
    vecfield( LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),Utdecoder(14,1)),
    

   }

In [97]:
for key,values in models_full.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

MiniViT model size: 0.13 MiB
MiniViT model params: 33185
MiniUNet model size: 0.13 MiB
MiniUNet model params: 33757
PCAMLP model size: 1.13 MiB
PCAMLP model params: 32938
MLP model size: 0.12 MiB
MLP model params: 31905


In [98]:
runner = FlowExperimentRunner(
    models=models_full,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1151.885: : 2500it [05:17,  7.87it/s, train=1151.8849, val=1156.8395]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1173.040: : 2500it [05:46,  7.22it/s, train=1173.0400, val=1168.6810]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1168.578: : 2500it [03:11, 13.04it/s, train=1168.5778, val=1173.0449]



>>> Training Model: MLP


Epoch 2499, loss: 1177.680: : 2500it [03:44, 11.12it/s, train=1177.6796, val=1171.4026]


In [99]:
save_complex_models({"models_full": models_full }, save_dir="saved_models-o")

Saving models_full / MiniViT to saved_models-o/models_full/MiniViT.pth
Saving models_full / MiniUNet to saved_models-o/models_full/MiniUNet.pth
Saving models_full / PCAMLP to saved_models-o/models_full/PCAMLP.pth
Saving models_full / MLP to saved_models-o/models_full/MLP.pth
✔️ Models saved to saved_models-o


## expr-2 latent ->inverse

In [43]:
latentmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}

In [44]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Forward model size: {model_size_b(forward_model) / MiB:.2f} MiB")
    print(f"  Forward model params: {count_model_params(forward_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(forward_model) + count_model_params(inverse_model)
    total_size=(model_size_b(forward_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")



[MiniViT]
  Forward model size: 0.06 MiB
  Forward model params: 16008
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 31832
  Total combined size: 0.13 MiB

[ConvNet]
  Forward model size: 0.06 MiB
  Forward model params: 16580
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 32976
  Total combined size: 0.13 MiB

[PCA_MLP]
  Forward model size: 1.06 MiB
  Forward model params: 15761
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 31611
  Total combined size: 2.13 MiB

[MLP]
  Forward model size: 0.06 MiB
  Forward model params: 16799
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 33437
  Total combined size: 0.13 MiB


In [45]:
runner = LatentFlowExperimentRunner(
    models=latentmodels,
    cfg_class=LatentCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


0it [00:00, ?it/s]

Epoch 4999, loss: 0.202: : 5000it [07:55, 10.53it/s, train=0.2016, val=0.1982]     



>>> Training Model: ConvNet


Epoch 4999, loss: 0.121: : 5000it [05:46, 14.45it/s, train=0.1213, val=0.1209]     



>>> Training Model: PCA_MLP


0it [00:00, ?it/s]/var/folders/xc/6hz9k5g17yxfwfktsvcnjr640000gn/T/ipykernel_7132/948465491.py:21: UserWarning: The operator 'aten::linalg_svd' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:14.)
  U, S, Vh = torch.linalg.svd(x_centered, full_matrices=False)
Epoch 4999, loss: 0.001: : 5000it [02:13, 37.48it/s, train=0.0015, val=0.0010]



>>> Training Model: MLP


Epoch 4999, loss: 0.000: : 5000it [02:11, 38.08it/s, train=0.0001, val=0.0002]


In [110]:
save_complex_models({"latentmodels": latentmodels}, save_dir="saved_models-o")

Saving latentmodels / MiniViT_forward to saved_models-o/latentmodels/MiniViT_forward.pth
Saving latentmodels / MiniViT_inverse to saved_models-o/latentmodels/MiniViT_inverse.pth
Saving latentmodels / ConvNet_forward to saved_models-o/latentmodels/ConvNet_forward.pth
Saving latentmodels / ConvNet_inverse to saved_models-o/latentmodels/ConvNet_inverse.pth
Saving latentmodels / PCA_MLP_forward to saved_models-o/latentmodels/PCA_MLP_forward.pth
Saving latentmodels / PCA_MLP_inverse to saved_models-o/latentmodels/PCA_MLP_inverse.pth
Saving latentmodels / MLP_forward to saved_models-o/latentmodels/MLP_forward.pth
Saving latentmodels / MLP_inverse to saved_models-o/latentmodels/MLP_inverse.pth
✔️ Models saved to saved_models-o


In [47]:
from torch.nn.functional import pairwise_distance,cosine_similarity

In [48]:
z, y = path.p_data.sample(1000) 
t = torch.rand(1000,1,1,1).to(z)
x = path.sample_conditional_path(z,t) 

ut_ref = path.conditional_vector_field(x,z,t) 

In [111]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    
    ut_forward =  forward_model(x,t,y) 
    ut_inverse = inverse_model(x)
    print(key)
    print('loss:',einsum( torch.square(ut_forward - ut_inverse),'b seq d ->b').mean())
    print('mse',F.mse_loss(ut_forward, ut_inverse))
    ut_forward=rearrange(ut_forward,'b seq d -> b (seq d)')
    ut_inverse=rearrange(ut_inverse,'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)

    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print('Wasserstein distance:', wasserstein_distance)


    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print('JS divergence:', js_divergence.mean())
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print('Cosine similarity:', cos_sim.mean())
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print('Bhattacharyya distance:', bhattacharyya_distance.mean())

MiniViT
loss: tensor(1.0283, device='mps:0', grad_fn=<MeanBackward0>)
mse tensor(0.0010, device='mps:0', grad_fn=<MseLossBackward0>)
Wasserstein distance: tensor(0.8748, device='mps:0', grad_fn=<MeanBackward0>)
JS divergence: tensor(0.0001, device='mps:0', grad_fn=<MeanBackward0>)
Cosine similarity: tensor(-0.0273, device='mps:0', grad_fn=<MeanBackward0>)
Bhattacharyya distance: tensor(0.0001, device='mps:0', grad_fn=<MeanBackward0>)
ConvNet
loss: tensor(0.0710, device='mps:0', grad_fn=<MeanBackward0>)
mse tensor(1.7327e-05, device='mps:0', grad_fn=<MseLossBackward0>)
Wasserstein distance: tensor(0.2378, device='mps:0', grad_fn=<MeanBackward0>)
JS divergence: tensor(-3.8814e-05, device='mps:0', grad_fn=<MeanBackward0>)
Cosine similarity: tensor(0.0050, device='mps:0', grad_fn=<MeanBackward0>)
Bhattacharyya distance: tensor(2.1665e-06, device='mps:0', grad_fn=<MeanBackward0>)
PCA_MLP
loss: tensor(0.0008, device='mps:0', grad_fn=<MeanBackward0>)
mse tensor(5.2509e-05, device='mps:0', gra

In [104]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

    # Reshape for probability-based metrics
    ut_forward = rearrange(ut_forward, 'b seq d -> b (seq d)')
    ut_inverse = rearrange(ut_inverse, 'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)

    # KL Divergence
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)
    print(f"  KL Divergence (forward): {kl_div_forward.mean()}")
    print(f"  KL Divergence (inverse): {kl_div_inverse.mean()}")

    # Wasserstein Distance
    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print(f"  Wasserstein Distance: {wasserstein_distance}")

    # JS Divergence
    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print(f"  JS Divergence: {js_divergence.mean()}")

    # Cosine Similarity
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print(f"  Cosine Similarity: {cos_sim.mean()}")

    # Bhattacharyya Distance
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print(f"  Bhattacharyya Distance: {bhattacharyya_distance.mean()}")
    
   

Model: MiniViT
  Loss: 1.0282931327819824
  MSE: 0.0009887435007840395
  KL Divergence (forward): 0.0004806879733223468
  KL Divergence (inverse): 0.0004813515115529299
  Wasserstein Distance: 0.8747981190681458
  JS Divergence: 0.00011239395826123655
  Cosine Similarity: -0.027275793254375458
  Bhattacharyya Distance: 0.0001228527253260836
Model: ConvNet
  Loss: 0.07096999138593674
  MSE: 1.7326661691186018e-05
  KL Divergence (forward): -3.232711969758384e-05
  KL Divergence (inverse): -3.232466406188905e-05
  Wasserstein Distance: 0.2377500981092453
  JS Divergence: -3.881366137648001e-05
  Cosine Similarity: 0.004964557942003012
  Bhattacharyya Distance: 2.166514832424582e-06
Model: PCA_MLP
  Loss: 0.0008401413797400892
  MSE: 5.250883987173438e-05
  KL Divergence (forward): 2.4512501113349572e-05
  KL Divergence (inverse): 2.4482640583300963e-05
  Wasserstein Distance: 0.02275516837835312
  JS Divergence: 6.014193786541e-06
  Cosine Similarity: -0.011280423030257225
  Bhattacharyy

In [106]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

    # Reshape for probability-based metrics
    ut_forward = rearrange(ut_forward, 'b seq d -> b (seq d)')
    ut_inverse = rearrange(ut_inverse, 'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)

    # KL Divergence
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)
    print(f"  KL Divergence (forward): {kl_div_forward.mean()}")
    print(f"  KL Divergence (inverse): {kl_div_inverse.mean()}")

    # Wasserstein Distance
    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print(f"  Wasserstein Distance: {wasserstein_distance}")

    # JS Divergence
    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print(f"  JS Divergence: {js_divergence.mean()}")

    # Cosine Similarity
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print(f"  Cosine Similarity: {cos_sim.mean()}")

    # Bhattacharyya Distance
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print(f"  Bhattacharyya Distance: {bhattacharyya_distance.mean()}")

Model: MiniViT
  Loss: 1.0282931327819824
  MSE: 0.0009887435007840395
  KL Divergence (forward): 0.0004806879733223468
  KL Divergence (inverse): 0.0004813515115529299
  Wasserstein Distance: 0.8747981190681458
  JS Divergence: 0.00011239395826123655
  Cosine Similarity: -0.027275793254375458
  Bhattacharyya Distance: 0.0001228527253260836
Model: ConvNet
  Loss: 0.07096999138593674
  MSE: 1.7326661691186018e-05
  KL Divergence (forward): -3.232711969758384e-05
  KL Divergence (inverse): -3.232466406188905e-05
  Wasserstein Distance: 0.2377500981092453
  JS Divergence: -3.881366137648001e-05
  Cosine Similarity: 0.004964557942003012
  Bhattacharyya Distance: 2.166514832424582e-06
Model: PCA_MLP
  Loss: 0.0008401413797400892
  MSE: 5.250883987173438e-05
  KL Divergence (forward): 2.4512501113349572e-05
  KL Divergence (inverse): 2.4482640583300963e-05
  Wasserstein Distance: 0.02275516837835312
  JS Divergence: 6.014193786541e-06
  Cosine Similarity: -0.011280423030257225
  Bhattacharyy

In [ ]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

    # Reshape for probability-based metrics
    ut_forward = rearrange(ut_forward, 'b seq d -> b (seq d)')
    ut_inverse = rearrange(ut_inverse, 'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)

    # KL Divergence
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)
    print(f"  KL Divergence (forward): {kl_div_forward.mean()}")
    print(f"  KL Divergence (inverse): {kl_div_inverse.mean()}")

    # Wasserstein Distance
    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print(f"  Wasserstein Distance: {wasserstein_distance}")

    # JS Divergence
    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print(f"  JS Divergence: {js_divergence.mean()}")

    # Cosine Similarity
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print(f"  Cosine Similarity: {cos_sim.mean()}")

    # Bhattacharyya Distance
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print(f"  Bhattacharyya Distance: {bhattacharyya_distance.mean()}")

## expr-3 inverse-decoder

In [51]:
inversemodels = {
    "MiniViT": {
        "decoder": Utdecoder(16,1),
        "inverse": copy.deepcopy(latentmodels['MiniViT']['inverse'])
    },
    "ConvNet": {
        "decoder":  Utdecoder(16,1),
        "inverse": copy.deepcopy(latentmodels['ConvNet']['inverse'])
    },
    "PCA_MLP": {
        "decoder": Utdecoder(16,1),
        "inverse": copy.deepcopy(latentmodels['PCA_MLP']['inverse'])
    },
    "MLP": {
        "decoder":Utdecoder(14,1),
        "inverse": copy.deepcopy(latentmodels['MLP']['inverse'])
    }
}

In [52]:
for key, models_pair in inversemodels.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Decoder model size: {model_size_b(decoder_model) / MiB:.2f} MiB")
    print(f"  Decoder model params: {count_model_params(decoder_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(decoder_model) + count_model_params(inverse_model)
    total_size=(model_size_b(decoder_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 33001
  Total combined size: 0.13 MiB

[ConvNet]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 33573
  Total combined size: 0.13 MiB

[PCA_MLP]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 33027
  Total combined size: 1.13 MiB

[MLP]
  Decoder model size: 0.06 MiB
  Decoder model params: 15106
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 31744
  Total combined size: 0.12 MiB


In [53]:


runner = InverseFlowExperimentRunner(
  models=inversemodels,
    cfg_class=InverseutCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3)
runner.run()



>>> Training Model: MiniViT


Epoch 4999, loss: 1304.839: : 5000it [06:25, 12.95it/s, train=1304.8395, val=1308.8551]



>>> Training Model: ConvNet


Epoch 4999, loss: 1363.504: : 5000it [06:13, 13.40it/s, train=1363.5037, val=1351.1752]



>>> Training Model: PCA_MLP


Epoch 4999, loss: 1331.278: : 5000it [04:16, 19.49it/s, train=1331.2776, val=1374.6799]



>>> Training Model: MLP


Epoch 4999, loss: 1306.142: : 5000it [03:31, 23.61it/s, train=1306.1425, val=1310.5427]  


In [54]:
save_complex_models({"inversemodels": inversemodels}, save_dir="saved_models-o")

Saving inversemodels / MiniViT_decoder to saved_models-o/inversemodels/MiniViT_decoder.pth
Saving inversemodels / MiniViT_inverse to saved_models-o/inversemodels/MiniViT_inverse.pth
Saving inversemodels / ConvNet_decoder to saved_models-o/inversemodels/ConvNet_decoder.pth
Saving inversemodels / ConvNet_inverse to saved_models-o/inversemodels/ConvNet_inverse.pth
Saving inversemodels / PCA_MLP_decoder to saved_models-o/inversemodels/PCA_MLP_decoder.pth
Saving inversemodels / PCA_MLP_inverse to saved_models-o/inversemodels/PCA_MLP_inverse.pth
Saving inversemodels / MLP_decoder to saved_models-o/inversemodels/MLP_decoder.pth
Saving inversemodels / MLP_inverse to saved_models-o/inversemodels/MLP_inverse.pth
✔️ Models saved to saved_models-o


## expr-4 combnine encoder decoder 

In [64]:
mms = {
    "MiniViT":  vecfield(copy.deepcopy(latentmodels['MiniViT']['forward']),copy.deepcopy(inversemodels['MiniViT']['decoder'])),
    
    "MiniUNet":vecfield(copy.deepcopy(latentmodels['ConvNet']['forward']),copy.deepcopy(inversemodels['ConvNet']['decoder'])),
    "PCAMLP": vecfield(copy.deepcopy(latentmodels['PCA_MLP']['forward']),copy.deepcopy(inversemodels['PCA_MLP']['decoder'])),
    "MLP":vecfield(copy.deepcopy(latentmodels['MLP']['forward']),copy.deepcopy(inversemodels['MLP']['decoder'])),
}

In [65]:
z, y = path.p_data.sample(1000) 
t = torch.rand(1000,1,1,1).to(z)
x = path.sample_conditional_path(z,t)

ut_ref = path.conditional_vector_field(x,z,t) 

In [66]:
for key, value in mms.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")
    

Model: MiniViT
  Loss: 1410.1146240234375
  MSE: 1.3770650625228882
  Cosine Similarity: 0.5173099637031555
Model: MiniUNet
  Loss: 1361.53955078125
  MSE: 1.3296284675598145
  Cosine Similarity: 0.5400453209877014
Model: PCAMLP
  Loss: 1426.6185302734375
  MSE: 1.3931822776794434
  Cosine Similarity: 0.522537112236023
Model: MLP
  Loss: 1742.4940185546875
  MSE: 1.701654314994812
  Cosine Similarity: 0.525293231010437


In [67]:
runner = FlowExperimentRunner(
    models=mms,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1244.592: : 2500it [04:57,  8.40it/s, train=1244.5917, val=1247.9756]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1215.858: : 2500it [04:57,  8.40it/s, train=1215.8584, val=1209.9573]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1220.304: : 2500it [03:02, 13.73it/s, train=1220.3038, val=1207.0710]



>>> Training Model: MLP


Epoch 2499, loss: 1262.287: : 2500it [03:03, 13.65it/s, train=1262.2867, val=1256.2485]


In [68]:
save_complex_models({"mms": mms}, save_dir="saved_models-o")

Saving mms / MiniViT to saved_models-o/mms/MiniViT.pth
Saving mms / MiniUNet to saved_models-o/mms/MiniUNet.pth
Saving mms / PCAMLP to saved_models-o/mms/PCAMLP.pth
Saving mms / MLP to saved_models-o/mms/MLP.pth
✔️ Models saved to saved_models-o


In [69]:
for key, value in mms.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")
    

Model: MiniViT
  Loss: 1235.252685546875
  MSE: 1.2063014507293701
  Cosine Similarity: 0.5975006222724915
Model: MiniUNet
  Loss: 1202.3345947265625
  MSE: 1.1741548776626587
  Cosine Similarity: 0.6113885045051575
Model: PCAMLP
  Loss: 1208.517333984375
  MSE: 1.1801927089691162
  Cosine Similarity: 0.609063446521759
Model: MLP
  Loss: 1256.5960693359375
  MSE: 1.2271445989608765
  Cosine Similarity: 0.5882124304771423


In [70]:
cms = {
    "MiniViT": {
        "forward":copy.deepcopy(latentmodels['MiniViT']['forward']) ,
        "inverse": copy.deepcopy(latentmodels['MiniViT']['inverse']),
        "decoder":copy.deepcopy(inversemodels['MiniViT']['decoder']),
    },
    "UNet": {
        "forward": copy.deepcopy(latentmodels['ConvNet']['forward']),
        "inverse":copy.deepcopy(latentmodels['ConvNet']['inverse']),
         "decoder":copy.deepcopy(inversemodels['ConvNet']['decoder']),
    },
    "PCA_MLP": {
         "forward": copy.deepcopy(latentmodels['PCA_MLP']['forward']),
        "inverse":copy.deepcopy(latentmodels['PCA_MLP']['inverse']),
         "decoder":copy.deepcopy(inversemodels['PCA_MLP']['decoder']),
    },
    "MLP": {
        "forward": copy.deepcopy(latentmodels['MLP']['forward']),
        "inverse":copy.deepcopy(latentmodels['MLP']['inverse']),
         "decoder":copy.deepcopy(inversemodels['MLP']['decoder']),
    }
}

In [71]:
for key, value in cms.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1410.1146240234375
  Loss inverse-decoder: 1301.56982421875
  Loss forward-inverse: 110.30496215820312
  MSE: 1.3770650625228882
  Cosine Similarity: 0.5173099637031555
Model: UNet
  Loss forward-decoder: 1361.53955078125
  Loss inverse-decoder: 1357.3267822265625
  Loss forward-inverse: 5.08932638168335
  MSE: 1.3296284675598145
  Cosine Similarity: 0.5400453209877014
Model: PCA_MLP
  Loss forward-decoder: 1426.6185302734375
  Loss inverse-decoder: 1341.3433837890625
  Loss forward-inverse: 84.33991241455078
  MSE: 1.3931822776794434
  Cosine Similarity: 0.522537112236023
Model: MLP
  Loss forward-decoder: 1742.4940185546875
  Loss inverse-decoder: 1305.860107421875
  Loss forward-inverse: 391.9115905761719
  MSE: 1.701654314994812
  Cosine Similarity: 0.525293231010437


In [72]:
runner = FullFlowExperimentRunner(
    models=cms,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=2500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 476.379: : 2500it [05:22,  7.76it/s, train=476.3785, val=475.0688]     



>>> Training Model: UNet


Epoch 2499, loss: 418.985: : 2500it [05:42,  7.30it/s, train=418.9855, val=418.5808]



>>> Training Model: PCA_MLP


Epoch 2499, loss: 544.462: : 2500it [04:36,  9.06it/s, train=544.4625, val=504.7098] 



>>> Training Model: MLP


Epoch 2499, loss: 428.462: : 2500it [04:17,  9.70it/s, train=428.4623, val=456.6604]  


In [73]:
save_complex_models({"cms": cms}, save_dir="saved_models-o")

Saving cms / MiniViT_forward to saved_models-o/cms/MiniViT_forward.pth
Saving cms / MiniViT_inverse to saved_models-o/cms/MiniViT_inverse.pth
Saving cms / MiniViT_decoder to saved_models-o/cms/MiniViT_decoder.pth
Saving cms / UNet_forward to saved_models-o/cms/UNet_forward.pth
Saving cms / UNet_inverse to saved_models-o/cms/UNet_inverse.pth
Saving cms / UNet_decoder to saved_models-o/cms/UNet_decoder.pth
Saving cms / PCA_MLP_forward to saved_models-o/cms/PCA_MLP_forward.pth
Saving cms / PCA_MLP_inverse to saved_models-o/cms/PCA_MLP_inverse.pth
Saving cms / PCA_MLP_decoder to saved_models-o/cms/PCA_MLP_decoder.pth
Saving cms / MLP_forward to saved_models-o/cms/MLP_forward.pth
Saving cms / MLP_inverse to saved_models-o/cms/MLP_inverse.pth
Saving cms / MLP_decoder to saved_models-o/cms/MLP_decoder.pth
✔️ Models saved to saved_models-o


In [74]:
for key, value in cms.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1310.4560546875
  Loss inverse-decoder: 1293.5457763671875
  Loss forward-inverse: 83.16475677490234
  MSE: 1.2797422409057617
  Cosine Similarity: 0.5649519562721252
Model: UNet
  Loss forward-decoder: 1251.14404296875
  Loss inverse-decoder: 1224.070556640625
  Loss forward-inverse: 13.819199562072754
  MSE: 1.22182035446167
  Cosine Similarity: 0.5905572772026062
Model: PCA_MLP
  Loss forward-decoder: 1382.288330078125
  Loss inverse-decoder: 1279.2647705078125
  Loss forward-inverse: 144.0518341064453
  MSE: 1.349890947341919
  Cosine Similarity: 0.5569304823875427
Model: MLP
  Loss forward-decoder: 1300.5693359375
  Loss inverse-decoder: 1275.5552978515625
  Loss forward-inverse: 26.736494064331055
  MSE: 1.2700872421264648
  Cosine Similarity: 0.5751382112503052


## for-inv-full

In [75]:
fullmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),
        "decoder":Utdecoder(16,1)
    },
    "UNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
         "decoder":Utdecoder(16,1)
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
         "decoder":Utdecoder(16,1)
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
         "decoder":Utdecoder(14,1) 
    }
}

In [76]:
runner = FullFlowExperimentRunner(
    models=fullmodels,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=2500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 605.186: : 2500it [05:27,  7.64it/s, train=605.1863, val=610.0003]



>>> Training Model: UNet


Epoch 2499, loss: 639.896: : 2500it [04:39,  8.95it/s, train=639.8964, val=640.2358]



>>> Training Model: PCA_MLP


Epoch 2499, loss: 494.124: : 2500it [02:20, 17.81it/s, train=494.1239, val=489.4894]



>>> Training Model: MLP


Epoch 2499, loss: 544.484: : 2500it [02:17, 18.13it/s, train=544.4843, val=542.4611]


In [77]:
save_complex_models({"fullmodels": fullmodels}, save_dir="saved_models-o")

Saving fullmodels / MiniViT_forward to saved_models-o/fullmodels/MiniViT_forward.pth
Saving fullmodels / MiniViT_inverse to saved_models-o/fullmodels/MiniViT_inverse.pth
Saving fullmodels / MiniViT_decoder to saved_models-o/fullmodels/MiniViT_decoder.pth
Saving fullmodels / UNet_forward to saved_models-o/fullmodels/UNet_forward.pth
Saving fullmodels / UNet_inverse to saved_models-o/fullmodels/UNet_inverse.pth
Saving fullmodels / UNet_decoder to saved_models-o/fullmodels/UNet_decoder.pth
Saving fullmodels / PCA_MLP_forward to saved_models-o/fullmodels/PCA_MLP_forward.pth
Saving fullmodels / PCA_MLP_inverse to saved_models-o/fullmodels/PCA_MLP_inverse.pth
Saving fullmodels / PCA_MLP_decoder to saved_models-o/fullmodels/PCA_MLP_decoder.pth
Saving fullmodels / MLP_forward to saved_models-o/fullmodels/MLP_forward.pth
Saving fullmodels / MLP_inverse to saved_models-o/fullmodels/MLP_inverse.pth
Saving fullmodels / MLP_decoder to saved_models-o/fullmodels/MLP_decoder.pth
✔️ Models saved to sav

In [78]:
ams= {
    "MiniViT":  vecfield(copy.deepcopy(fullmodels['MiniViT']['forward']),copy.deepcopy(fullmodels['MiniViT']['decoder'])),
    
    "MiniUNet":vecfield(copy.deepcopy(fullmodels['UNet']['forward']),copy.deepcopy(fullmodels['UNet']['decoder'])),
    "PCAMLP": vecfield(copy.deepcopy(fullmodels['PCA_MLP']['forward']),copy.deepcopy(fullmodels['PCA_MLP']['decoder'])),
    "MLP":vecfield(copy.deepcopy(fullmodels['MLP']['forward']),copy.deepcopy(fullmodels['MLP']['decoder'])),
}

In [79]:
for key, value in fullmodels.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1760.24267578125
  Loss inverse-decoder: 1738.4700927734375
  Loss forward-inverse: 1.0230176448822021
  MSE: 1.718986988067627
  Cosine Similarity: 0.5182914137840271
Model: UNet
  Loss forward-decoder: 1920.6787109375
  Loss inverse-decoder: 1916.7451171875
  Loss forward-inverse: 0.007069090381264687
  MSE: 1.8756628036499023
  Cosine Similarity: 0.29898470640182495
Model: PCA_MLP
  Loss forward-decoder: 1414.883056640625
  Loss inverse-decoder: 1389.3470458984375
  Loss forward-inverse: 10.722254753112793
  MSE: 1.3817217350006104
  Cosine Similarity: 0.5155596137046814
Model: MLP
  Loss forward-decoder: 1593.47119140625
  Loss inverse-decoder: 1564.205322265625
  Loss forward-inverse: 6.987271785736084
  MSE: 1.556124210357666
  Cosine Similarity: 0.4225931763648987


In [80]:
for key, value in ams.items():
    
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1760.24267578125
  MSE: 1.718986988067627
  Cosine Similarity: 0.5182914137840271
Model: MiniUNet
  Loss: 1920.6787109375
  MSE: 1.8756628036499023
  Cosine Similarity: 0.29898470640182495
Model: PCAMLP
  Loss: 1414.883056640625
  MSE: 1.3817217350006104
  Cosine Similarity: 0.5155596137046814
Model: MLP
  Loss: 1593.47119140625
  MSE: 1.556124210357666
  Cosine Similarity: 0.4225931763648987


## expr inverse path

In [50]:
inversemodels2 = {
    "MiniViT": {
        "decoder": Utdecoder(16,1),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "decoder":  Utdecoder(16,1),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32)
    },
    "PCA_MLP": {
        "decoder": Utdecoder(16,1),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256)
    },
    "MLP": {
        "decoder":Utdecoder(14,1),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}



In [51]:
for key, models_pair in inversemodels2.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Decoder model size: {model_size_b(decoder_model) / MiB:.2f} MiB")
    print(f"  Decoder model params: {count_model_params(decoder_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(decoder_model) + count_model_params(inverse_model)
    total_size=(model_size_b(decoder_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 33001
  Total combined size: 0.13 MiB

[ConvNet]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 33573
  Total combined size: 0.13 MiB

[PCA_MLP]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 33027
  Total combined size: 1.13 MiB

[MLP]
  Decoder model size: 0.06 MiB
  Decoder model params: 15106
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 31744
  Total combined size: 0.12 MiB


In [52]:

runner = InverseFlowExperimentRunner2(
  models=inversemodels2,
    cfg_class=InverseutCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3)
runner.run()


>>> Training Model: MiniViT


0it [00:00, ?it/s]

Epoch 4999, loss: 1158.676: : 5000it [08:36,  9.69it/s, train=1158.6757, val=1145.5549]



>>> Training Model: ConvNet


Epoch 2121, loss: 1215.973: : 2122it [02:36, 13.58it/s, train=1215.9734, val=1203.1719]


KeyboardInterrupt: 

In [113]:
save_complex_models({"inversemodels2": inversemodels2}, save_dir="saved_models-o")

Saving inversemodels2 / MiniViT_decoder to saved_models-o/inversemodels2/MiniViT_decoder.pth
Saving inversemodels2 / MiniViT_inverse to saved_models-o/inversemodels2/MiniViT_inverse.pth
Saving inversemodels2 / ConvNet_decoder to saved_models-o/inversemodels2/ConvNet_decoder.pth
Saving inversemodels2 / ConvNet_inverse to saved_models-o/inversemodels2/ConvNet_inverse.pth
Saving inversemodels2 / PCA_MLP_decoder to saved_models-o/inversemodels2/PCA_MLP_decoder.pth
Saving inversemodels2 / PCA_MLP_inverse to saved_models-o/inversemodels2/PCA_MLP_inverse.pth
Saving inversemodels2 / MLP_decoder to saved_models-o/inversemodels2/MLP_decoder.pth
Saving inversemodels2 / MLP_inverse to saved_models-o/inversemodels2/MLP_inverse.pth
✔️ Models saved to saved_models-o


In [53]:
for key, models_pair in inversemodels2.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    ut_inverse = inverse_model(ut_ref)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    cos_sim = cosine_similarity(
        rearrange(ut_theta_i, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

NameError: name 'ut_ref' is not defined

In [54]:
latentmodels2 = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": copy.deepcopy(inversemodels2["MiniViT"]['inverse'])
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": copy.deepcopy(inversemodels2["ConvNet"]['inverse'])
                ,
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse":copy.deepcopy(inversemodels2["PCA_MLP"]['inverse'])
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse":copy.deepcopy(inversemodels2["MLP"]['inverse'])
    }
}




In [116]:
for key, models_pair in latentmodels2.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Forward model size: {model_size_b(forward_model) / MiB:.2f} MiB")
    print(f"  Forward model params: {count_model_params(forward_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(forward_model) + count_model_params(inverse_model)
    total_size=(model_size_b(forward_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Forward model size: 0.06 MiB
  Forward model params: 16008
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 31832
  Total combined size: 0.13 MiB

[ConvNet]
  Forward model size: 0.06 MiB
  Forward model params: 16580
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 32976
  Total combined size: 0.13 MiB

[PCA_MLP]
  Forward model size: 1.06 MiB
  Forward model params: 15761
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 31611
  Total combined size: 2.13 MiB

[MLP]
  Forward model size: 0.06 MiB
  Forward model params: 16799
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 33437
  Total combined size: 0.13 MiB


In [55]:
runner = LatentFlowExperimentRunner2(
    models=latentmodels2,
    cfg_class=LatentCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 4999, loss: 529.233: : 5000it [07:57, 10.48it/s, train=529.2330, val=578.4211]   



>>> Training Model: ConvNet


Epoch 4999, loss: 424.449: : 5000it [06:57, 11.97it/s, train=424.4485, val=430.9575]   



>>> Training Model: PCA_MLP


0it [00:00, ?it/s]/var/folders/xc/6hz9k5g17yxfwfktsvcnjr640000gn/T/ipykernel_37063/948465491.py:21: UserWarning: The operator 'aten::linalg_svd' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:14.)
  U, S, Vh = torch.linalg.svd(x_centered, full_matrices=False)
Epoch 4999, loss: 12.825: : 5000it [59:06,  1.41it/s, train=12.8246, val=12.5698]



>>> Training Model: MLP


Epoch 4999, loss: 4.935: : 5000it [02:14, 37.27it/s, train=4.9353, val=5.0096]   


In [118]:
save_complex_models({"latentmodels2": latentmodels2}, save_dir="saved_models-o")

Saving latentmodels2 / MiniViT_forward to saved_models-o/latentmodels2/MiniViT_forward.pth
Saving latentmodels2 / MiniViT_inverse to saved_models-o/latentmodels2/MiniViT_inverse.pth
Saving latentmodels2 / ConvNet_forward to saved_models-o/latentmodels2/ConvNet_forward.pth
Saving latentmodels2 / ConvNet_inverse to saved_models-o/latentmodels2/ConvNet_inverse.pth
Saving latentmodels2 / PCA_MLP_forward to saved_models-o/latentmodels2/PCA_MLP_forward.pth
Saving latentmodels2 / PCA_MLP_inverse to saved_models-o/latentmodels2/PCA_MLP_inverse.pth
Saving latentmodels2 / MLP_forward to saved_models-o/latentmodels2/MLP_forward.pth
Saving latentmodels2 / MLP_inverse to saved_models-o/latentmodels2/MLP_inverse.pth
✔️ Models saved to saved_models-o


In [56]:
for key, models_pair in latentmodels2.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

    # Reshape for probability-based metrics
    ut_forward = rearrange(ut_forward, 'b seq d -> b (seq d)')
    ut_inverse = rearrange(ut_inverse, 'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)

    # KL Divergence
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)
    print(f"  KL Divergence (forward): {kl_div_forward.mean()}")
    print(f"  KL Divergence (inverse): {kl_div_inverse.mean()}")

    # Wasserstein Distance
    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print(f"  Wasserstein Distance: {wasserstein_distance}")

    # JS Divergence
    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print(f"  JS Divergence: {js_divergence.mean()}")

    # Cosine Similarity
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print(f"  Cosine Similarity: {cos_sim.mean()}")

    # Bhattacharyya Distance
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print(f"  Bhattacharyya Distance: {bhattacharyya_distance.mean()}")

NameError: name 'x' is not defined

In [91]:
mms2 = {
    "MiniViT":  vecfield(copy.deepcopy(latentmodels2['MiniViT']['forward']),copy.deepcopy(inversemodels2['MiniViT']['decoder'])),
    
    "MiniUNet":vecfield(copy.deepcopy(latentmodels2['ConvNet']['forward']),copy.deepcopy(inversemodels2['ConvNet']['decoder'])),
    "PCAMLP": vecfield(latentmodels2['PCA_MLP']['forward'],inversemodels2['PCA_MLP']['decoder']),
    "MLP":vecfield(copy.deepcopy(latentmodels2['MLP']['forward']),copy.deepcopy(inversemodels2['MLP']['decoder'])),
}

In [92]:
for key, value in mms2.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1293.9765625
  MSE: 1.2636489868164062
  Cosine Similarity: 0.5723602771759033
Model: MiniUNet


  Loss: 1225.3072509765625
  MSE: 1.1965891122817993
  Cosine Similarity: 0.6015927791595459
Model: PCAMLP
  Loss: 1226.46435546875
  MSE: 1.1977190971374512
  Cosine Similarity: 0.601493775844574
Model: MLP
  Loss: 1255.8138427734375
  MSE: 1.226380705833435
  Cosine Similarity: 0.5888971090316772


In [93]:
runner = FlowExperimentRunner(
    models=mms2,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1149.215: : 2500it [05:18,  7.85it/s, train=1149.2151, val=1136.4591]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1123.794: : 2500it [06:02,  6.89it/s, train=1123.7936, val=1118.6908]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1172.283: : 2500it [03:05, 13.47it/s, train=1172.2828, val=1164.7616]



>>> Training Model: MLP


Epoch 2499, loss: 1157.545: : 2500it [03:29, 11.95it/s, train=1157.5450, val=1166.9105]


In [94]:
save_complex_models({"mms2": mms2}, save_dir="saved_models-o")

Saving mms2 / MiniViT to saved_models-o/mms2/MiniViT.pth
Saving mms2 / MiniUNet to saved_models-o/mms2/MiniUNet.pth
Saving mms2 / PCAMLP to saved_models-o/mms2/PCAMLP.pth
Saving mms2 / MLP to saved_models-o/mms2/MLP.pth
✔️ Models saved to saved_models-o


In [95]:
for key, value in mms2.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1142.83056640625
  MSE: 1.116045594215393
  Cosine Similarity: 0.6362442970275879
Model: MiniUNet
  Loss: 1114.1473388671875
  MSE: 1.0880345106124878
  Cosine Similarity: 0.6480246782302856
Model: PCAMLP
  Loss: 1161.832763671875
  MSE: 1.1346023082733154
  Cosine Similarity: 0.6284326314926147
Model: MLP
  Loss: 1158.76806640625
  MSE: 1.1316094398498535
  Cosine Similarity: 0.6296930313110352


# save

In [93]:
import os

In [161]:
models_group = {
    "modelo": modelo,
    "models_full": models_full,
    "latentmodels": latentmodels,
    "inversemodels": inversemodels,
    "mms": mms,
    "cms": cms,
    "inversemodels2":inversemodels2,
    "latentmodels2":latentmodels2,
      "mms2": mms2,
    
    
}

In [95]:
latentmodels3 = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}

In [77]:
models_group = {
   
    "latentmodels": latentmodels,
    
    
}

In [162]:

def save_complex_models(nested_dict, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)

    for top_key, sub_dict in nested_dict.items():
        model_folder = os.path.join(save_dir, top_key)
        os.makedirs(model_folder, exist_ok=True)

        if isinstance(sub_dict, dict):
            for name, model in sub_dict.items():
               
                if isinstance(model, dict):
                    for subname, submodel in model.items():
                        path = os.path.join(model_folder, f"{name}_{subname}.pth")
                        print(f"Saving {top_key} / {name}_{subname} to {path}")
                        torch.save(submodel.state_dict(), path)
                else:
                    path = os.path.join(model_folder, f"{name}.pth")
                    print(f"Saving {top_key} / {name} to {path}")
                    torch.save(model.state_dict(), path)
        else:
          
            path = os.path.join(save_dir, f"{top_key}.pth")
            print(f"Saving {top_key} to {path}")
            torch.save(sub_dict.state_dict(), path)

    print(f"✔️ Models saved to {save_dir}")


In [163]:
save_complex_models(models_group, save_dir="saved_models-o")

Saving modelo / MiniViT to saved_models-o/modelo/MiniViT.pth
Saving modelo / MiniUNet to saved_models-o/modelo/MiniUNet.pth
Saving modelo / PCAMLP to saved_models-o/modelo/PCAMLP.pth
Saving modelo / MLP to saved_models-o/modelo/MLP.pth
Saving models_full / MiniViT to saved_models-o/models_full/MiniViT.pth
Saving models_full / MiniUNet to saved_models-o/models_full/MiniUNet.pth
Saving models_full / PCAMLP to saved_models-o/models_full/PCAMLP.pth
Saving models_full / MLP to saved_models-o/models_full/MLP.pth
Saving latentmodels / MiniViT_forward to saved_models-o/latentmodels/MiniViT_forward.pth
Saving latentmodels / MiniViT_inverse to saved_models-o/latentmodels/MiniViT_inverse.pth
Saving latentmodels / ConvNet_forward to saved_models-o/latentmodels/ConvNet_forward.pth
Saving latentmodels / ConvNet_inverse to saved_models-o/latentmodels/ConvNet_inverse.pth
Saving latentmodels / PCA_MLP_forward to saved_models-o/latentmodels/PCA_MLP_forward.pth
Saving latentmodels / PCA_MLP_inverse to sa

✅ Loaded modelo/MiniViT
✅ Loaded modelo/MiniUNet
✅ Loaded modelo/PCAMLP
✅ Loaded modelo/MLP
✅ Loaded models_full/MiniViT
✅ Loaded models_full/MiniUNet
✅ Loaded models_full/PCAMLP
✅ Loaded models_full/MLP
✅ Loaded latentmodels/MiniViT_forward
✅ Loaded latentmodels/MiniViT_inverse
✅ Loaded latentmodels/ConvNet_forward
✅ Loaded latentmodels/ConvNet_inverse
✅ Loaded latentmodels/PCA_MLP_forward
✅ Loaded latentmodels/PCA_MLP_inverse
✅ Loaded latentmodels/MLP_forward
✅ Loaded latentmodels/MLP_inverse
✅ Loaded inversemodels/MiniViT_decoder
✅ Loaded inversemodels/MiniViT_inverse
✅ Loaded inversemodels/ConvNet_decoder
✅ Loaded inversemodels/ConvNet_inverse
✅ Loaded inversemodels/PCA_MLP_decoder
✅ Loaded inversemodels/PCA_MLP_inverse
✅ Loaded inversemodels/MLP_decoder
✅ Loaded inversemodels/MLP_inverse
✅ Loaded mms/MiniViT
✅ Loaded mms/MiniUNet
✅ Loaded mms/PCAMLP
✅ Loaded mms/MLP
✅ Loaded cms/MiniViT_forward
✅ Loaded cms/MiniViT_inverse
✅ Loaded cms/MiniViT_decoder
✅ Loaded cms/UNet_forward
✅ L

In [164]:
def load_all_model_families(model_families, save_dir="saved_vecfield_models"):

    loaded_families = {}

    for family_name, model_dict in model_families.items():
        folder_path = os.path.join(save_dir, family_name)
        if not os.path.exists(folder_path):
            print(f"❌ folder {folder_path} not found, skipping.")
            continue

        loaded_families[family_name] = {}

        for model_name, model_obj in model_dict.items():
            if isinstance(model_obj, dict):
                loaded_families[family_name][model_name] = {}
                for subname, submodel in model_obj.items():
                    filename = f"{model_name}_{subname}.pth"
                    path = os.path.join(folder_path, filename)
                    if os.path.exists(path):
                        submodel.load_state_dict(torch.load(path, map_location='cpu'))
                        submodel.eval()
                        loaded_families[family_name][model_name][subname] = submodel
                        print(f"✅ Loaded {family_name}/{model_name}_{subname}")
                    else:
                        print(f"❌ Missing: {path}")
            else:
                filename = f"{model_name}.pth"
                path = os.path.join(folder_path, filename)
                if os.path.exists(path):
                    model_obj.load_state_dict(torch.load(path, map_location='cpu'))
                    model_obj.eval()
                    loaded_families[family_name][model_name] = model_obj
                    print(f"✅ Loaded {family_name}/{model_name}")
                else:
                    print(f"❌ Missing: {path}")

    return loaded_families

In [174]:
loaded_models = load_all_model_families({
   "modelo": modelo,
    "models_full": models_full,
    "latentmodels": latentmodels,
    "inversemodels": inversemodels,
    "mms": mms,
    "cms": cms,
    "inversemodels2":inversemodels2,
    "latentmodels2":latentmodels2,
      "mms2": mms2,
}, save_dir="saved_models-o")

✅ Loaded modelo/MiniViT
✅ Loaded modelo/MiniUNet
✅ Loaded modelo/PCAMLP
✅ Loaded modelo/MLP
✅ Loaded models_full/MiniViT
✅ Loaded models_full/MiniUNet
✅ Loaded models_full/PCAMLP
✅ Loaded models_full/MLP
✅ Loaded latentmodels/MiniViT_forward
✅ Loaded latentmodels/MiniViT_inverse
✅ Loaded latentmodels/ConvNet_forward
✅ Loaded latentmodels/ConvNet_inverse
✅ Loaded latentmodels/PCA_MLP_forward
✅ Loaded latentmodels/PCA_MLP_inverse
✅ Loaded latentmodels/MLP_forward
✅ Loaded latentmodels/MLP_inverse
✅ Loaded inversemodels/MiniViT_decoder
✅ Loaded inversemodels/MiniViT_inverse
✅ Loaded inversemodels/ConvNet_decoder
✅ Loaded inversemodels/ConvNet_inverse
✅ Loaded inversemodels/PCA_MLP_decoder
✅ Loaded inversemodels/PCA_MLP_inverse
✅ Loaded inversemodels/MLP_decoder
✅ Loaded inversemodels/MLP_inverse
✅ Loaded mms/MiniViT
✅ Loaded mms/MiniUNet
✅ Loaded mms/PCAMLP
✅ Loaded mms/MLP
✅ Loaded cms/MiniViT_forward
✅ Loaded cms/MiniViT_inverse
✅ Loaded cms/MiniViT_decoder
✅ Loaded cms/UNet_forward
✅ L

In [167]:
loaded_models = load_all_model_families({
    "latentmodels": latentmodels3,

}, save_dir="saved_models-o")

✅ Loaded latentmodels/MiniViT_forward
✅ Loaded latentmodels/MiniViT_inverse
✅ Loaded latentmodels/ConvNet_forward
✅ Loaded latentmodels/ConvNet_inverse
✅ Loaded latentmodels/PCA_MLP_forward
✅ Loaded latentmodels/PCA_MLP_inverse
✅ Loaded latentmodels/MLP_forward
✅ Loaded latentmodels/MLP_inverse


In [180]:
id(latentmodels3) ==id(loaded_models['latentmodels'])

False

In [178]:
for key, models_pair in loaded_models['latentmodels'].items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    forward_model.to(device).eval()
    inverse_model.to(device).eval()
    

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")

Model: MiniViT
  Loss: 0.5283627510070801
Model: ConvNet
  Loss: 0.056214042007923126
Model: PCA_MLP
  Loss: 0.0021014667581766844
Model: MLP
  Loss: 1.1761752367019653


# load

## func

In [92]:
def load_all_model_families(model_families, save_dir="saved_vecfield_models"):

    loaded_families = {}

    for family_name, model_dict in model_families.items():
        folder_path = os.path.join(save_dir, family_name)
        if not os.path.exists(folder_path):
            print(f"❌ folder {folder_path} not found, skipping.")
            continue

        loaded_families[family_name] = {}

        for model_name, model_obj in model_dict.items():
            if isinstance(model_obj, dict):
                loaded_families[family_name][model_name] = {}
                for subname, submodel in model_obj.items():
                    filename = f"{model_name}_{subname}.pth"
                    path = os.path.join(folder_path, filename)
                    if os.path.exists(path):
                        submodel.load_state_dict(torch.load(path, map_location='cpu'))
                        submodel.eval()
                        loaded_families[family_name][model_name][subname] = submodel
                        print(f"✅ Loaded {family_name}/{model_name}_{subname}")
                    else:
                        print(f"❌ Missing: {path}")
            else:
                filename = f"{model_name}.pth"
                path = os.path.join(folder_path, filename)
                if os.path.exists(path):
                    model_obj.load_state_dict(torch.load(path, map_location='cpu'))
                    model_obj.eval()
                    loaded_families[family_name][model_name] = model_obj
                    print(f"✅ Loaded {family_name}/{model_name}")
                else:
                    print(f"❌ Missing: {path}")

    return loaded_families

## models

In [ ]:
modelo = {
    "MiniViT": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32
                ),
                selector=TopKSelector(d_model=16, k=8)
            ),
            out_channels=1
        )
    ),
    "MiniUNet": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
                selector=TopKSelector(d_model=16, k=8)  #
            ),
            out_channels=1
        )
    ),
    "PCAMLP": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ),
                selector=TopKSelector(d_model=16, k=8)
            ),
            out_channels=1
        )
    ),
    "MLP": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
                selector=TopKSelector(d_model=14, k=8)
            ),
            out_channels=1
        )
    ),
}

models_full={ "MiniViT":
    vecfield( LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),Utdecoder(16,1)),
    "MiniUNet":
    vecfield( LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),Utdecoder(16,1)),
    "PCAMLP":
    vecfield(LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),Utdecoder(16,1)),
    "MLP":
    vecfield( LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),Utdecoder(14,1)),
    

   }
latentmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}
inversemodels = {
    "MiniViT": {
        "decoder": Utdecoder(16,1),
        "inverse": latentmodels['MiniViT']['inverse']
    },
    "ConvNet": {
        "decoder":  Utdecoder(16,1),
        "inverse": latentmodels['ConvNet']['inverse']
    },
    "PCA_MLP": {
        "decoder": Utdecoder(16,1),
        "inverse": latentmodels['PCA_MLP']['inverse']
    },
    "MLP": {
        "decoder":Utdecoder(14,1),
        "inverse": latentmodels['MLP']['inverse']
    }
}
mms = {
    "MiniViT":  vecfield(latentmodels['MiniViT']['forward'],inversemodels['MiniViT']['decoder']),
    
    "MiniUNet":vecfield(latentmodels['ConvNet']['forward'],inversemodels['ConvNet']['decoder']),
    "PCAMLP": vecfield(latentmodels['PCA_MLP']['forward'],inversemodels['PCA_MLP']['decoder']),
    "MLP":vecfield(latentmodels['MLP']['forward'],inversemodels['MLP']['decoder']),
}
cms = {
    "MiniViT": {
        "forward":latentmodels['MiniViT']['forward'] ,
        "inverse": latentmodels['MiniViT']['inverse'],
        "decoder":inversemodels['MiniViT']['decoder'],
    },
    "UNet": {
        "forward": latentmodels['ConvNet']['forward'],
        "inverse":latentmodels['ConvNet']['inverse'],
         "decoder":inversemodels['ConvNet']['decoder'],
    },
    "PCA_MLP": {
         "forward": latentmodels['PCA_MLP']['forward'],
        "inverse":latentmodels['PCA_MLP']['inverse'],
         "decoder":inversemodels['PCA_MLP']['decoder'],
    },
    "MLP": {
        "forward": latentmodels['MLP']['forward'],
        "inverse":latentmodels['MLP']['inverse'],
         "decoder":inversemodels['MLP']['decoder'],
    }
}
fullmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),
        "decoder":Utdecoder(16,1)
    },
    "UNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
         "decoder":Utdecoder(16,1)
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
         "decoder":Utdecoder(16,1)
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
         "decoder":Utdecoder(14,1) 
    }
}
ams= {
    "MiniViT":  vecfield(fullmodels['MiniViT']['forward'],fullmodels['MiniViT']['decoder']),
    
    "MiniUNet":vecfield(fullmodels['UNet']['forward'],fullmodels['UNet']['decoder']),
    "PCAMLP": vecfield(fullmodels['PCA_MLP']['forward'],fullmodels['PCA_MLP']['decoder']),
    "MLP":vecfield(fullmodels['MLP']['forward'],fullmodels['MLP']['decoder']),
}

inversemodels2 = {
    "MiniViT": {
        "decoder": Utdecoder(16,1),
        "inverse": latentmodels['MiniViT']['inverse']
    },
    "ConvNet": {
        "decoder":  Utdecoder(16,1),
        "inverse": latentmodels['ConvNet']['inverse']
    },
    "PCA_MLP": {
        "decoder": Utdecoder(16,1),
        "inverse": latentmodels['PCA_MLP']['inverse']
    },
    "MLP": {
        "decoder":Utdecoder(14,1),
        "inverse": latentmodels['MLP']['inverse']
    }
}
latentmodels2 = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}
mms2 = {
    "MiniViT":  vecfield(latentmodels2['MiniViT']['forward'],inversemodels2['MiniViT']['decoder']),
    
    "MiniUNet":vecfield(latentmodels2['ConvNet']['forward'],inversemodels2['ConvNet']['decoder']),
    "PCAMLP": vecfield(latentmodels2['PCA_MLP']['forward'],inversemodels2['PCA_MLP']['decoder']),
    "MLP":vecfield(latentmodels2['MLP']['forward'],inversemodels2['MLP']['decoder']),
}

## load

In [ ]:
models_group = {
    "modelo": modelo,
    "models_full": models_full,
    "latentmodels": latentmodels,
    "inversemodels": inversemodels,
    "mms": mms,
    "cms": cms,
    "inversemodels2":inversemodels2,
    "latentmodels2":latentmodels2,
      "mms2": mms2,
    
    
}

loaded_models = load_all_model_families({
   "modelo": modelo,
    "models_full": models_full,
    "latentmodels": latentmodels,
    "inversemodels": inversemodels,
    "mms": mms,
    "cms": cms,
    "inversemodels2":inversemodels2,
    "latentmodels2":latentmodels2,
      "mms2": mms2,
}, save_dir="saved_models-o")

In [ ]:
from torchvision.utils import make_grid

In [ ]:
# Play with these!
samples_per_class = 10
num_timesteps = 100
guidance_scales = [1.0, 3.0, 5.0]

# Graph
fig, axes = plt.subplots(1, len(guidance_scales), figsize=(10 * len(guidance_scales), 10))

for idx, w in enumerate(guidance_scales):
    # Setup ode and simulator
    ode = CFGVectorFieldODE(net, guidance_scale=w)
    simulator = EulerSimulator(ode)

    # Sample initial conditions
    y = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=torch.int64).repeat_interleave(samples_per_class).to(device)
    num_samples = y.shape[0]
    x0, _ = path.p_simple.sample(num_samples) # (num_samples, 1, 32, 32)

    # Simulate
    ts = torch.linspace(0,1,num_timesteps).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1).to(device)
    x1 = simulator.simulate(x0, ts, y=y)

    # Plot
    grid = make_grid(x1, nrow=samples_per_class, normalize=True, value_range=(-1,1))
    axes[idx].imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    axes[idx].axis("off")
    axes[idx].set_title(f"Guidance: $w={w:.1f}$", fontsize=25)

In [ ]:
net=mms['MiniViT']
net2=mms['MiniUNet']
net3=mms['PCAMLP']
net4=mms['MLP']

In [ ]:
y = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=torch.int64).repeat_interleave(4).to(device)
num_samples = y.shape[0]
x0, _ = path.p_simple.sample(num_samples) # (num_samples, 1, 32, 32)
ts = torch.linspace(0,1,num_timesteps).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1).to(device)

In [ ]:
simulator1 = EulerSimulator( CFGVectorFieldODE(net, guidance_scale=1.0))
simulator2 = EulerSimulator( CFGVectorFieldODE(net2, guidance_scale=1.0))
simulator3 = EulerSimulator( CFGVectorFieldODE(net3, guidance_scale=1.0))
simulator4 = EulerSimulator( CFGVectorFieldODE(net4, guidance_scale=1.0))

In [ ]:
x11 = simulator1.simulate(x0, ts, y=y)
x12=simulator2.simulate(x0, ts, y=y)
x13=simulator3.simulate(x0, ts, y=y)
x14=simulator4.simulate(x0, ts, y=y)

In [ ]:
einsum(torch.square(x11),'b c h w->b')

In [ ]:
# Play with these!
samples_per_class = 10
num_timesteps = 100
guidance_scales = [1.0, 3.0, 5.0]

# Graph
fig, axes = plt.subplots(1, len(guidance_scales), figsize=(10 * len(guidance_scales), 10))

for idx, w in enumerate(guidance_scales):
    # Setup ode and simulator
    ode = CFGVectorFieldODE(net2, guidance_scale=w)
    simulator = EulerSimulator(ode)

    # Sample initial conditions
    y = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=torch.int64).repeat_interleave(samples_per_class).to(device)
    num_samples = y.shape[0]
    x0, _ = path.p_simple.sample(num_samples) # (num_samples, 1, 32, 32)

    # Simulate
    ts = torch.linspace(0,1,num_timesteps).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1).to(device)
    x1 = simulator.simulate(x0, ts, y=y)

    # Plot
    grid = make_grid(x1, nrow=samples_per_class, normalize=True, value_range=(-1,1))
    axes[idx].imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    axes[idx].axis("off")
    axes[idx].set_title(f"Guidance: $w={w:.1f}$", fontsize=25)

In [ ]:
net3=mms['PCAMLP']

In [ ]:
# Play with these!
samples_per_class = 10
num_timesteps = 100
guidance_scales = [1.0, 3.0, 5.0]

# Graph
fig, axes = plt.subplots(1, len(guidance_scales), figsize=(10 * len(guidance_scales), 10))

for idx, w in enumerate(guidance_scales):
    # Setup ode and simulator
    ode = CFGVectorFieldODE(net3, guidance_scale=w)
    simulator = EulerSimulator(ode)

    # Sample initial conditions
    y = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=torch.int64).repeat_interleave(samples_per_class).to(device)
    num_samples = y.shape[0]
    x0, _ = path.p_simple.sample(num_samples) # (num_samples, 1, 32, 32)

    # Simulate
    ts = torch.linspace(0,1,num_timesteps).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1).to(device)
    x1 = simulator.simulate(x0, ts, y=y)

    # Plot
    grid = make_grid(x1, nrow=samples_per_class, normalize=True, value_range=(-1,1))
    axes[idx].imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    axes[idx].axis("off")
    axes[idx].set_title(f"Guidance: $w={w:.1f}$", fontsize=25)

In [ ]:
fullmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),
        "decoder":Utdecoder(16,1)
    },
    "UNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
         "decoder":Utdecoder(16,1)
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
         "decoder":Utdecoder(16,1)
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
         "decoder":Utdecoder(14,1) 
    }
}

In [ ]:
path = GaussianConditionalProbabilityPath(
    p_data = MNISTSampler(),
    p_simple_shape = [1, 32, 32],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

In [ ]:
runner = FullFlowExperimentRunner(
    models=fullmodels,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()

In [ ]:
class FullCFG(nn.Module):
    def __init__(self,forward_model,inverse_model,decoder_model,path: GaussianConditionalProbabilityPath, eta: float,):
      super().__init__()
      self.eta=eta
      self.path=path
      self.forward_model=forward_model
      self.inverse_model=inverse_model
      self.decoder_model=decoder_model
    
    def forward( self,batch_size: int)->Float[Array,'...']:
        z, y = self.path.p_data.sample(batch_size)
        xi = torch.rand(y.shape[0]).to(y.device)
        y[xi < self.eta] = 10.0
        
        t = torch.rand(batch_size,1,1,1).to(z) 
        x = self.path.sample_conditional_path(z,t) 
        
        
        ut_theta_latent = self.forward_model(x,t,y) 
        ut_ref = self.path.conditional_vector_field(x,z,t)
        ut_ref_latent=self.inverse_model(ut_ref)
        ut_inverse_path=self.decoder_model(ut_ref_latent)
        ut_forward_path=self.decoder_model(ut_theta_latent)
        #loss_latent=einsum(torch.square(ut_theta_latent-ut_ref_latent),'b seq d ->b').mean()
        loss_inverse_path=einsum(torch.square(ut_inverse_path- ut_ref),'b c h w ->b').mean()
        loss_forward_path=einsum(torch.square(ut_forward_path- ut_ref),'b c h w ->b').mean()
        #loss_diff_path= einsum(torch.square(ut_forward_path-ut_inverse_path),'b c h w ->b').mean()
        #error = einsum(torch.square( ut_theta_latent-ut_ref_latent),'b seq d -> b')
        return (loss_inverse_path+ loss_forward_path)/2

In [ ]:
runner = FullFlowExperimentRunner(
    models=fullmodels,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()

In [ ]:
class FullCFG(nn.Module):
    def __init__(self,forward_model,inverse_model,decoder_model,path: GaussianConditionalProbabilityPath, eta: float,):
      super().__init__()
      self.eta=eta
      self.path=path
      self.forward_model=forward_model
      self.inverse_model=inverse_model
      self.decoder_model=decoder_model
    
    def forward( self,batch_size: int)->Float[Array,'...']:
        z, y = self.path.p_data.sample(batch_size)
        xi = torch.rand(y.shape[0]).to(y.device)
        y[xi < self.eta] = 10.0
        
        t = torch.rand(batch_size,1,1,1).to(z) 
        x = self.path.sample_conditional_path(z,t) 
        
        
        ut_theta_latent = self.forward_model(x,t,y) 
        ut_ref = self.path.conditional_vector_field(x,z,t)
        ut_ref_latent=self.inverse_model(ut_ref)
        ut_inverse_path=self.decoder_model(ut_ref_latent)
        ut_forward_path=self.decoder_model(ut_theta_latent)
        loss_latent=einsum(torch.square(ut_theta_latent-ut_ref_latent),'b seq d ->b').mean()
        loss_inverse_path=einsum(torch.square(ut_inverse_path- ut_ref),'b c h w ->b').mean()
        loss_forward_path=einsum(torch.square(ut_forward_path- ut_ref),'b c h w ->b').mean()
        loss_diff_path= einsum(torch.square(ut_forward_path-ut_inverse_path),'b c h w ->b').mean()
        #error = einsum(torch.square( ut_theta_latent-ut_ref_latent),'b seq d -> b')
        return (loss_inverse_path+ loss_forward_path)/2+(loss_latent+loss_diff_path)/2

In [ ]:
device

In [ ]:
runner = FullFlowExperimentRunner(
    models=fullmodels,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=1000,
    batch_size=1000,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()

In [ ]:
runner = FullFlowExperimentRunner(
    models=fullmodels,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()

In [ ]:
inverse_cfg_loss =InverseCFG (
                decoder_model=decoder_model,
                inverse_model=inverse_model,
                path=path,
                eta=0.01
            )


In [ ]:
decoder_model=inversemodels['MiniViT']['decoder']
inverse_model=inversemodels['MiniViT']['inverse']

In [ ]:
inverse_cfg_loss(64)

In [ ]:
inversemodels = {
    "MiniViT": {
        "decoder": Udecoder(16,1),
        "inverse": models['MiniViT']['inverse']
    },
    "ConvNet": {
        "decoder":  Udecoder(16,1),
        "inverse": models['ConvNet']['inverse']
    },
    "PCA_MLP": {
        "decoder": Udecoder(16,1),
        "inverse": models['PCA_MLP']['inverse']
    },
    "MLP": {
        "decoder":Udecoder(14,1),
        "inverse": models['MLP']['inverse']
    }
}

In [ ]:
for key, models_pair in inversemodels.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
   
    print('decoder',id(decoder_model))
    print('inverse',id(inverse_model))

In [ ]:
class biencoder(nn.Module):
    def __init__(self,encoder,invencoder):
        self.encoder=encoder
        self.invencoder=invencoder
        
    def forward(self, x: Float[Array,"bs c h w"], t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]) -> Float[Array,"bs c h w "]:
        x=self.encoder
       
        x = self.invencoder(x)
    
        return x
    
    
    

In [ ]:
m1=models['MiniViT']['forward'].to(device)
m2=inversemodels['MiniViT']['decoder'].to(device)
m3=inversemodels['MiniViT']['inverse'].to(device)

In [ ]:
z, y = path.p_data.sample(32)
t = torch.rand(32,1,1,1).to(z)
x = path.sample_conditional_path(z,t)
ut_ref = path.conditional_vector_field(x,z,t)

In [ ]:
einsum(torch.square(m2(m1(x,t,y))-ut_ref),'b c h w -> b' )

In [ ]:
z, y = path.p_data.sample(32)
t = torch.rand(32,1,1,1).to(z)
x = path.sample_conditional_path(z,t)
ut_ref = path.conditional_vector_field(x,z,t)
                                    

In [ ]:
print(id(trainer.decoder_model ))


In [ ]:
m1.eval()
m2.eval()
m3.eval()

In [ ]:
inverse_cfg_loss =InverseCFG (
                decoder_model=m2,
                inverse_model=m3,
                path=path,
                eta=0.01
            )

In [ ]:
inverse_cfg_loss =InverseCFG (
                decoder_model=m2,
                inverse_model=m3,
                path=path,
                eta=0.01
            )
C

In [ ]:
latent_cfg_loss =LatentCFG(
               forward_model=m1,
                inverse_model=m3,
                path=path,
                eta=0.01
            )

In [ ]:
inverse_cfg_loss(32)

In [ ]:
latent_cfg_loss(32)

In [ ]:
u1=m1(x,t,y)
u2=m2(u1)



In [ ]:
u2.shape,ut_ref.shape

In [ ]:
torch.mean(einsum(torch.square(u2 - ut_ref),'b c h w -> b') )

In [ ]:
m1(x,t,y).shape,m3(ut_ref).shape

In [ ]:
torch.mean(einsum(torch.square(m1(x,t,y) - m3(ut_ref)),'b seq d-> b') )

In [ ]:
torch.mean(einsum(torch.square(m2(m1(x,t,y)) - ut_ref),'b c h w-> b') )

In [ ]:
torch.mean(einsum(torch.square(m2(m3(x)) - ut_ref),'b c h w-> b') )

In [ ]:
torch.mean(einsum(torch.square(u2 - ut_ref),'b c h w -> b') )

In [ ]:
    latent=self.inverse_model(ut_ref)
        ut_decode=self.decoder_model(latent)
        error = einsum(torch.square( ut_decode- ut_ref),'b c h w -> b')

In [ ]:
torch.mean(einsum(torch.square(m3(ut_ref) - u1),'b seq d -> b') )

In [ ]:
torch.mean(einsum(torch.square(m2(m3(ut_ref)) -ut_ref ),'b c h w-> b') )

In [ ]:
m3(ut_ref)-u1

In [ ]:
u1=m1(x,t,y)

In [ ]:
u1=m1(x,t,y)
u2=m2(u1)

torch.mean(einsum(torch.square(u2 - ut_ref),'b c h w -> b') )

In [ ]:
u2=m2(u1)

In [ ]:
ut_theta = u2

In [ ]:
error=einsum(torch.square(ut_theta - ut_ref),'b c h w -> b') 

In [ ]:
torch.mean(error)

In [ ]:
ut_ref_latent.size(-1)

In [ ]:
t_decoder=Udecoder(ut_ref_latent.size(-1),1).to(device)

In [ ]:
t_decoder(ut_ref_latent).shape

In [ ]:
error=einsum(torch.square(ut_ref-t_decoder(ut_ref_latent)),'b c h w -> b')
torch.mean(error)


In [ ]:
ut_ref-t_decoder(ut_ref_latent)

In [ ]:
kl_gaussian_per_sample(ut_theta_latent.mean(2),ut_ref_latent.mean(2))

In [ ]:
kl_gaussian_per_sample(ut_ref_latent.mean(2),ut_theta_latent.mean(2))

In [ ]:
torch.log(F.layer_norm(ut_theta_latent,ut_theta_latent.shape[1:]))

In [ ]:
torch.log(F.layer_norm(ut_theta_latent,ut_theta_latent.shape[1:])+1e-8)

In [ ]:
ut_theta_latent.log()

In [ ]:
torch.log(ut_theta_latent).

In [ ]:
def kl_continuous(p: Float[Array.'b seq d'], q:Float[Array.'b seq d'])->:
  
    p_mean, p_std = p.mean(0), p.std(0)
    q_mean, q_std = q.mean(0), q.std(0)
    kl = torch.log(q_std / p_std) + (p_std**2 + (p_mean - q_mean)**2) / (2 * q_std**2) - 0.5
    return kl.sum()

In [ ]:
forward_model=LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16)
inverse_model=MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)

loss_model=LatentCFG(forward_model,inverse_model,path=path,eta=0.1)

In [ ]:
runner = FlowExperimentRunner(
    models=models,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()

In [ ]:
loss=loss_model(64)

In [ ]:
torch.square(x_t -inverse_model(ut_ref)).shape

In [ ]:
error = einsum(torch.square(x_t -inverse_model(ut_ref)),'b seq d -> b')


In [ ]:
torch.mean(einsum((x_t -inverse_model(ut_ref)),'b seq d -> b'))

In [ ]:
error

In [ ]:
torch.mean(error)

In [ ]:
ex=torch.randn(10,1,32,32)

In [ ]:
z,y=path.p_data.sample(10)
t=torch.rand(10,1,1,1)
x = path.sample_conditional_path(z,t)


In [ ]:
 t: Float[Array,"bs 1 1 1"], y: Float[Array,"bs ..."]

In [ ]:
m1(x,t=t,y=y).shape

In [ ]:
ut_ref = path.conditional_vector_field(x,z,t)

In [ ]:
for key,values in models.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

In [ ]:
runner = FlowExperimentRunner(
    models=models,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()

In [ ]:
import os

def save_vecfield_models(models, save_dir="saved_vecfield_models"):
    os.makedirs(save_dir, exist_ok=True)

    for name, vecfield in models.items():
        matcher = vecfield.matcher
        latent_model = matcher.latent_model
        extractor = latent_model.extractor
        selector = latent_model.selector

      
        save_dict = {
            'vecfield_class': type(vecfield).__name__,

            'matcher_class': type(matcher).__name__,
            'matcher_init': {
                'd_in': matcher.t_embeder.half_dim * 2,  # Corrected to access half_dim
                'out_channels': matcher.docoder[-1].out_channels if hasattr(matcher.docoder[-1], 'out_channels') else None, # Corrected to use out_channels
            },

            'extractor_class': type(extractor).__name__,
            'extractor_init': {
                'd_in': extractor.d_in,
                'd_out': extractor.d_out,
                'd_hidden': getattr(extractor, 'd_hidden', None),
                'patch_size': getattr(extractor, 'patch_size', None),
                'num_heads': getattr(extractor, 'num_heads', None),
                'img_size': getattr(extractor, 'img_size', None),
                'n_components': getattr(extractor, 'n_components', None),
                'kernel_num': getattr(extractor, 'kernel_num', None),
            },
            'extractor_state_dict': extractor.state_dict(),

            'selector_class': type(selector).__name__,
            'selector_init': {
                'd_model': extractor.d_out,
                'k': selector.k,
            },
            'selector_state_dict': selector.state_dict(),

            'matcher_state_dict': matcher.state_dict(),
            'vecfield_state_dict': vecfield.state_dict()
        }

        
        save_dict['extractor_init'] = {
            k: v for k, v in save_dict['extractor_init'].items() if v is not None
        }

        save_dict['matcher_init'] = {
            k: v for k, v in save_dict['matcher_init'].items() if v is not None
        }

        save_path = os.path.join(save_dir, f"{name}_vecfield.pth")
        torch.save(save_dict, save_path)
        print(f"[✓] Saved {name} to {save_path}")

In [ ]:
def save_vecfield_models(models, save_dir="saved_vecfield_models"):
    os.makedirs(save_dir, exist_ok=True)

    for name, vecfield in models.items():
        matcher = vecfield.matcher
        latent_model = matcher.latent_model
        extractor = latent_model.extractor
        selector = latent_model.selector

        def extract_init_args(obj, arg_names):
            init_args = {}
            for arg in arg_names:
                if hasattr(obj, arg):
                    init_args[arg] = getattr(obj, arg)
            return init_args

        save_dict = {
            'vecfield_class': type(vecfield).__name__,

            'matcher_class': type(matcher).__name__,
            'matcher_init': {
                'd_in': matcher.d_in,
                'out_channels': matcher.out_channels,
            },

            'extractor_class': type(extractor).__name__,
            'extractor_init': extract_init_args(
                extractor,
                ['d_in', 'd_out', 'd_hidden', 'patch_size', 'num_heads', 'img_size', 'n_components', 'kernel_num']
            ),
            'extractor_state_dict': extractor.state_dict(),

            'selector_class': type(selector).__name__,
            'selector_init': {
                'd_model': selector.d_model,
                'k': selector.k,
            },
            'selector_state_dict': selector.state_dict(),

            'matcher_state_dict': matcher.state_dict(),
            'vecfield_state_dict': vecfield.state_dict()
        }

        save_path = os.path.join(save_dir, f"{name}_vecfield.pth")
        torch.save(save_dict, save_path)
        print(f"[✓] Saved {name} to {save_path}")

In [ ]:
save_vecfield_models(models)